#### Description of the Data

In [ ]:
# Formatting of plots

import matplotlib.pyplot as plt
import matplotlib as mpl
import scienceplots
import os 

plt.style.use(['science', 'nature','bright'])
mpl.rcParams['savefig.format'] = 'svg'
os.makedirs('plots/efus2017/southwest_bedroom_4month', exist_ok=True)
os.makedirs('plots/efus2017/southwest_bedroom_1week', exist_ok=True)


In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

ROOT = os.path.abspath('.')

# Extract room name for later
path = f"{ROOT}/efus_indoor_outdoor_livingroom.parquet"
df = pd.read_parquet(path)
room_name = str(path.split("/")[-1].replace(".parquet", "").split("_")[-1])

# Load parquet file 
print("Loading temperature data for",room_name,"...")
df = pd.read_parquet(f"{ROOT}/efus_indoor_outdoor_livingroom.parquet")

# Filter out extreme values
df = df[df["T_in"].between(-10, 40) & df["T_out"].between(-10, 40)]

# Add building characteristic data
BLDG_COLS = ["CaseID","dwtype_efus","dwage_efus","WallType2x_efus",
             "InsulatedWalls_efus","FullyDblGlz_efus","floor6x_efus","EPceeb12e_efus",
             "gorEHS_efus","AnyCooling"]
bldg = pd.read_csv(
    f"{ROOT}/test_data/ukda_9434_csv_r/csv/selected_interview_responses_caseid.csv",
    usecols=BLDG_COLS)
df = df.merge(bldg, on="CaseID", how="left").dropna(subset=BLDG_COLS[1:])

# Nominal
for c in BLDG_COLS[1:]:
    df[c] = df[c].astype(int)

# Filter to Southwest subset 
df = df[df["gorEHS_efus"] == 8].copy()
print(f"Southwest: {df['CaseID'].nunique()} dwellings, {len(df):,} observations")

# Filter to May-August
df = df[df['hour'].dt.month.between(5, 8)]

# Derived features
T_OUT_MEAN = df["T_out"].mean()
T_OUT_SD   = df["T_out"].std()

# Centered T_out on global mean across all dwellings and times
df["T_out_c"]     = df["T_out"] - T_OUT_MEAN
df["hour_of_day"] = df["hour"].dt.hour

# Descriptive stats 
df["T_out_q95"] = df['T_out'].quantile(0.95)
df["T_out_q5"] = df['T_out'].quantile(0.05)
median = df["T_out"].median()
q05 = df["T_out"].quantile(0.05)
q25 = df["T_out"].quantile(0.25)
q75 = df["T_out"].quantile(0.75)
q95 = df["T_out"].quantile(0.95)
iqr = q75 - q25

# Population-level diurnal terms
df["sin_h"] = np.sin(2 * np.pi * df["hour_of_day"] / 24)
df["cos_h"] = np.cos(2 * np.pi * df["hour_of_day"] / 24)

# At least 20 measurements over the measurement period
sizes = df.groupby("CaseID").size()
df_m = df[df["CaseID"].isin(sizes[sizes >= 20].index)].reset_index(drop=True)
df_m["dwelling"] = df_m["CaseID"].astype(str)

n_obs = len(df_m)
n_dwells = df_m['dwelling'].nunique()
r = int(n_obs/n_dwells/7)

print(f"\nModel dataset           : {n_obs} observations, {n_dwells} dwellings ({r} obs/dwells/week)")
print(f"T_out                   : median = {median:.1f}°C [IQR {iqr:.1f}°C]")
print(f"T_out                   : mean = {T_OUT_MEAN:.2f}°C,  SD = {T_OUT_SD:.2f}°C")
print(f"T_out central 90% range : {q05:.1f}–{q95:.1f}°C")

df.describe()

In [ ]:
# Building characteristics 

# One row per dwelling
dwelling_chars = (
    df_m[[
        "CaseID",
        "dwtype_efus",
        "dwage_efus",
        "WallType2x_efus",
        "InsulatedWalls_efus",
        "FullyDblGlz_efus",
        "floor6x_efus",
        "EPceeb12e_efus",
        "AnyCooling"
    ]]
    .drop_duplicates()
)

# Labels from efus
LEVELS = {

    "dwtype_efus": {
        1: "Detached",
        2: "Semi-detached",
        3: "End-terrace",
        4: "Mid-terrace",
        5: "Bungalow",
        6: "Flat/maisonette"
    },

    "dwage_efus": {
        1: "pre-1919",
        2: "1919–1944",
        3: "1945–1964",
        4: "1965–1974",
        5: "1975–1980",
        6: "1981–1990",
        7: "post-1990"
    },

    "WallType2x_efus": {
        1: "Solid wall",
        2: "Cavity wall"
    },

    "InsulatedWalls_efus": {
        0: "No",
        1: "Yes"
    },

    "FullyDblGlz_efus": {
        0: "No",
        1: "Yes"
    },

    "floor6x_efus": {
        1: r"$<50$ sqm",
        2: "50 - 69 sqm",
        3: "70 - 89 sqm",
        4: "90 - 109 sqm",
        5: "110 - 139 sqm",
        6: r"$>140$ sqm"
    },

    "EPceeb12e_efus": {
        1: "$>$ 70 (C+)",
        2: "30 - 50 (D)",
        3: "51 - 70 (E)",
        4: "$<$ 30 (F/G)"
    },

    "AnyCooling": {
        0: "No",
        1: "Yes"
    }
}

print(f"\nBuilding characteristics ({len(dwelling_chars)} dwellings)\n")

for col in LEVELS.keys():

    counts = dwelling_chars[col].value_counts().sort_index()
    perc = 100 * counts / counts.sum()

    summary = pd.DataFrame({
        "code": counts.index,
        "level": [LEVELS[col].get(i, "Unknown") for i in counts.index],
        "count": counts.values,
        "percent": perc.values.round(1)
    })

    print(f"\n{col}")
    print(summary.to_string(index=False))

# Most common archetype 
archetype_cols = [
    "dwtype_efus",
    "dwage_efus",
    "WallType2x_efus",
    "InsulatedWalls_efus",
    "FullyDblGlz_efus",
    "floor6x_efus",
    "EPceeb12e_efus",
    "AnyCooling"
]

archetypes = (
    dwelling_chars[archetype_cols]
    .value_counts()
    .reset_index(name="count")
)

top = archetypes.iloc[0]
print("\nMost common dwelling archetype\n")

for col in archetype_cols:
    label = LEVELS[col].get(top[col], str(top[col]))
    print(f"{col:22s}: {label}")

print(f"\nCount: {top['count']} dwellings")
print("\nMost common characteristics\n")

for col in archetype_cols:
    # Most frequent code
    mode_code = dwelling_chars[col].mode()[0]
    # Count
    count = (dwelling_chars[col] == mode_code).sum()
    # Percentage
    percent = 100 * count / len(dwelling_chars)
    # Label
    label = LEVELS[col].get(mode_code, str(mode_code))
    print(
        f"{col:22s}: {label:20s} "
        f"({count} dwellings, {percent:.1f}%)"
    )

In [ ]:
import matplotlib.dates as mdates
import textwrap

def plot_temps(dwelling_ids):

    # Ensure iterable
    if isinstance(dwelling_ids, str):
        dwelling_ids = [dwelling_ids]

    n = len(dwelling_ids)
    ncols = 2
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(12, 4.8 * nrows),
        squeeze=False
    )

    axes = axes.flatten()

    for ax, random_dwelling in zip(axes, dwelling_ids):

        # Subset data for that dwelling
        df_plot = (
            df_m[df_m["dwelling"] == str(random_dwelling)]
            .sort_values("hour")
        )

        # Building characteristics
        row = df_plot.iloc[0]

        char_text = (
            f"{LEVELS['dwtype_efus'][row['dwtype_efus']]}, "
            f"{LEVELS['dwage_efus'][row['dwage_efus']]}, "
            f"{LEVELS['WallType2x_efus'][row['WallType2x_efus']]}, "
            f"Insulated walls: "
            f"{LEVELS['InsulatedWalls_efus'][row['InsulatedWalls_efus']]}, "
            f"Double glazing: "
            f"{LEVELS['FullyDblGlz_efus'][row['FullyDblGlz_efus']]}, "
            f"Floor: {LEVELS['floor6x_efus'][row['floor6x_efus']]}, "
            f"EPC: {LEVELS['EPceeb12e_efus'][row['EPceeb12e_efus']]}, "
            f"Cooling: {LEVELS['AnyCooling'][row['AnyCooling']]}"
        )

        # Wrap title text to subplot width
        wrapped_text = textwrap.fill(char_text, width=45)

        ax.plot(
            df_plot["hour"],
            df_plot["T_out"],
            marker="x",
            markersize=0,
            label=r"$T_{out}$"
        )

        ax.plot(
            df_plot["hour"],
            df_plot["T_out_c"],
            marker="x",
            markersize=0,
            label=r"$T_{out,\ centered}$"
        )

        ax.plot(
            df_plot["hour"],
            df_plot["T_in"],
            marker="x",
            markersize=0,
            label=r"$T_{in}$"
        )

        ax.fill_between(
            df_plot["hour"],
            27,
            29,
            color='black',
            alpha=0.2
        )

        # Format x-axis as MM-DD only
        ax.xaxis.set_major_formatter(
            mdates.DateFormatter("%m-%d")
        )

        ax.set_xlabel("Date")
        ax.set_ylabel(r"Temperature ($^\circ$C)")

        ax.set_title(
            f"Dwelling {random_dwelling} ({room_name})\n"
            f"{wrapped_text}",
            fontsize=8,
            pad=12,
            loc="center"
        )

        ax.legend(fontsize=8)

    # Remove unused axes
    for ax in axes[n:]:
        fig.delaxes(ax)

    # Extra spacing between subplots
    fig.subplots_adjust(
        hspace=0.5,
        wspace=0.1
    )

    plt.savefig(
        f"plots/efus2017/southwest_bedroom_4month/"
        f"T_out_multiple_{room_name}.svg",
        bbox_inches="tight",
        dpi=300
    )

    plt.show()


# plot_temps(["1784", "310"])
plot_temps([df_m["dwelling"].unique()[0]])

In [ ]:
# EPC T_{in} distribution violin plot 

import matplotlib.colors as mcolors

# EPC labels
EPC_AXIS_LABELS = {
    1: "C+\n(SAP $>70$)",
    2: "D\n(SAP 51–70)",
    3: "E\n(SAP 30–50)",
    4: "F/G\n(SAP $<30$)",
}

display_bands = [b for b in [4, 3, 2, 1] if (df_m["EPceeb12e_efus"] == b).any()]

# Prepare EPC dataset
df_epc = df_m[["CaseID", "T_in", "EPceeb12e_efus"]].copy()

_all_data    = [df_epc[df_epc["EPceeb12e_efus"] == b]["T_in"].values for b in display_bands]
_all_labels  = [EPC_AXIS_LABELS[b] for b in display_bands]
_mask        = [len(d) > 0 for d in _all_data]
hourly_data  = [d for d, m in zip(_all_data, _mask) if m]
xlabels      = [l for l, m in zip(_all_labels, _mask) if m]


# Plot
fig, ax = plt.subplots(figsize=(3.4, 2.5))

parts = ax.violinplot(
    hourly_data,
    positions=range(1, len(hourly_data) + 1),
    showmeans=False,
    showmedians=False,
    showextrema=True
)

# Style violins
dark_colors = []
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

for i, pc in enumerate(parts["bodies"]):

    color = colors[i % len(colors)]
    dark_color = tuple(np.array(mcolors.to_rgb(color)) * 0.45)

    dark_colors.append(dark_color)

    pc.set_facecolor(color)
    pc.set_edgecolor(color)
    pc.set_alpha(0.45)

# Keep only central spine
parts["cmins"].set_visible(False)
parts["cmaxes"].set_visible(False)
parts["cbars"].set_linewidth(1.2)
parts["cbars"].set_color(dark_colors)

# Add IQR box, median point, and labels
for i, data in enumerate(hourly_data, start=1):

    color = colors[i - 1]
    dark_color = dark_colors[i - 1]

    q1, med, q3 = np.percentile(data, [25, 50, 75])

    ax.add_patch(
        plt.Rectangle(
            (i - 0.06, q1),
            0.12,
            q3 - q1,
            facecolor=color,
            edgecolor=dark_color,
            linewidth=1.2,
            zorder=4
        )
    )

    # ax.scatter(
    #     i,
    #     med,
    #     color="white",
    #     edgecolor=dark_color,
    #     linewidth=0.6,
    #     s=26,
    #     zorder=5
    # )

    ax.plot(
        [i - 0.049, i + 0.049],
        [med, med],
        color="white",
        linewidth=2.2,
        solid_capstyle="butt",
        zorder=5
    )

    # Median label (aligned with median dot)
    ax.text(
        i + 0.23,
        med,
        f"{med:.1f}°C",
        fontsize=5,
        va="center",
        ha="center",
        color=dark_color
    )

    # IQR label (left side of violin, vertical)
    # Position text on right except for last violin
    x_offset = 0.4 if i < len(hourly_data) else - 0.4
    y_offset = q1 - 0.5 if  i < len(hourly_data) else q3 + 0.5
    ax.text(
        i + x_offset,
        y_offset,
        f"IQR\n[{q1:.1f}, {q3:.1f}] °C",
        fontsize=5,
        va="top",
        ha="center",
        color=dark_color
    )

# Formatting
ymin = min(np.min(data) for data in hourly_data) - 1
ymax = max(np.max(data) for data in hourly_data) + 1
ax.set_ylim(ymin, ymax)
ax.set_xticks(range(1, len(hourly_data) + 1))
ax.set_xticklabels(xlabels)
ax.tick_params(axis='x', which='minor', bottom=False, top=False)
ax.set_xlabel("EPC score band")
ax.set_ylabel(r"Indoor temperature, $T_{in}$ ($^\circ$C)")

ax.set_title(
    "Indoor"f" {room_name} ""temperature distributions by EPC score band\n"
    "EFUS 2017 Southwest subset, May-August"
)

ax.grid(axis="y")

plt.tight_layout()
plt.savefig(f"plots/efus2017/southwest_bedroom_4month/epc_tin_distribution_{room_name}.svg", bbox_inches="tight")
plt.show()


In [ ]:
# AnyCooling T_{in} distribution violin plot

import matplotlib.colors as mcolors

# Cooling labels
COOLING_AXIS_LABELS = {
    0: "No\nCooling",
    1: "Cooling"
}

display_groups = [0, 1]

# Prepare dataset
df_cooling = df_m[["CaseID", "T_in", "AnyCooling"]].copy()

hourly_data = [
    df_cooling[df_cooling["AnyCooling"] == g]["T_in"].values
    for g in display_groups
]

xlabels = [
    COOLING_AXIS_LABELS[g]
    for g in display_groups
]

# Plot
fig, ax = plt.subplots(figsize=(2.6, 2.5))

parts = ax.violinplot(
    hourly_data,
    positions=range(1, len(display_groups) + 1),
    showmeans=False,
    showmedians=False,
    showextrema=True
)

# Style violins
dark_colors = []
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

for i, pc in enumerate(parts["bodies"]):

    color = colors[i % len(colors)]

    dark_color = tuple(
        np.array(mcolors.to_rgb(color)) * 0.45
    )

    dark_colors.append(dark_color)

    pc.set_facecolor(color)
    pc.set_edgecolor(color)
    pc.set_alpha(0.45)

# Keep only central spine
parts["cmins"].set_visible(False)
parts["cmaxes"].set_visible(False)

parts["cbars"].set_linewidth(1.2)
parts["cbars"].set_color(dark_colors)

# Add IQR box, median line, and labels
for i, data in enumerate(hourly_data, start=1):

    color = colors[i - 1]
    dark_color = dark_colors[i - 1]

    q1, med, q3 = np.percentile(
        data,
        [25, 50, 75]
    )

    ax.add_patch(
        plt.Rectangle(
            (i - 0.06, q1),
            0.12,
            q3 - q1,
            facecolor=color,
            edgecolor=dark_color,
            linewidth=1.2,
            zorder=4
        )
    )

    ax.plot(
        [i - 0.049, i + 0.049],
        [med, med],
        color="white",
        linewidth=2.2,
        solid_capstyle="butt",
        zorder=5
    )

    # Median label
    ax.text(
        i + 0.23,
        med,
        f"{med:.1f}°C",
        fontsize=5,
        va="center",
        ha="center",
        color=dark_color
    )

    # IQR label
    x_offset = 0.38 if i == 1 else -0.38
    y_offset = q1 - 0.5 if i == 1 else q3 + 0.5

    ax.text(
        i + x_offset,
        y_offset,
        f"IQR\n[{q1:.1f}, {q3:.1f}] °C",
        fontsize=5,
        va="top",
        ha="center",
        color=dark_color
    )

# Formatting
ymin = min(np.min(data) for data in hourly_data) - 1
ymax = max(np.max(data) for data in hourly_data) + 1

ax.set_ylim(ymin, ymax)

ax.set_xticks(
    range(1, len(display_groups) + 1)
)

ax.set_xticklabels(xlabels)

ax.tick_params(
    axis='x',
    which='minor',
    bottom=False,
    top=False
)

ax.set_xlabel("Cooling presence")

ax.set_ylabel(
    r"Indoor temperature, $T_{in}$ ($^\circ$C)"
)

ax.set_title(
    f"Indoor {room_name} temperature distributions by cooling presence\n"
    "EFUS 2017 Southwest subset, May-August"
)

ax.grid(axis="y")

plt.tight_layout()

plt.savefig(
    f"plots/efus2017/southwest_bedroom_4month/"
    f"cooling_tin_distribution_{room_name}.svg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# EPC T_{out_c} distribution violin plot 

import matplotlib.colors as mcolors

# EPC labels
EPC_AXIS_LABELS = {
    1: "C+\n(SAP $>70$)",
    2: "D\n(SAP 51–70)",
    3: "E\n(SAP 30–50)",
    4: "F/G\n(SAP $<30$)",
}

display_bands = [b for b in [4, 3, 2, 1] if (df_m["EPceeb12e_efus"] == b).any()]

# Prepare EPC dataset
df_epc = df_m[["CaseID", "T_out_c", "EPceeb12e_efus"]].copy()

_all_data    = [df_epc[df_epc["EPceeb12e_efus"] == b]["T_out_c"].values for b in display_bands]
_all_labels  = [EPC_AXIS_LABELS[b] for b in display_bands]
_mask        = [len(d) > 0 for d in _all_data]
hourly_data  = [d for d, m in zip(_all_data, _mask) if m]
xlabels      = [l for l, m in zip(_all_labels, _mask) if m]


# Plot
fig, ax = plt.subplots(figsize=(3.4, 2.5))

parts = ax.violinplot(
    hourly_data,
    positions=range(1, len(hourly_data) + 1),
    showmeans=False,
    showmedians=False,
    showextrema=True
)

# Style violins
dark_colors = []
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

for i, pc in enumerate(parts["bodies"]):

    color = colors[i % len(colors)]
    dark_color = tuple(np.array(mcolors.to_rgb(color)) * 0.45)

    dark_colors.append(dark_color)

    pc.set_facecolor(color)
    pc.set_edgecolor(color)
    pc.set_alpha(0.45)

# Keep only central spine
parts["cmins"].set_visible(False)
parts["cmaxes"].set_visible(False)
parts["cbars"].set_linewidth(1.2)
parts["cbars"].set_color(dark_colors)

# Add IQR box, median point, and labels
for i, data in enumerate(hourly_data, start=1):

    color = colors[i - 1]
    dark_color = dark_colors[i - 1]

    q1, med, q3 = np.percentile(data, [25, 50, 75])

    ax.add_patch(
        plt.Rectangle(
            (i - 0.06, q1),
            0.12,
            q3 - q1,
            facecolor=color,
            edgecolor=dark_color,
            linewidth=1.2,
            zorder=4
        )
    )

    # ax.scatter(
    #     i,
    #     med,
    #     color="white",
    #     edgecolor=dark_color,
    #     linewidth=0.6,
    #     s=26,
    #     zorder=5
    # )

    ax.plot(
        [i - 0.049, i + 0.049],
        [med, med],
        color="white",
        linewidth=2.2,
        solid_capstyle="butt",
        zorder=5
    )

    # Median label (aligned with median dot)
    ax.text(
        i + 0.23,
        med,
        f"{med:.1f}°C",
        fontsize=5,
        va="center",
        ha="center",
        color=dark_color
    )

    # IQR label (left side of violin, vertical)
    # Position text on right except for last violin
    x_offset = 0.4 if i < len(hourly_data) else - 0.4
    y_offset = q1 - 0.5 if  i < len(hourly_data) else q3 + 0.5
    ax.text(
        i + x_offset,
        y_offset,
        f"IQR\n[{q1:.1f}, {q3:.1f}] °C",
        fontsize=5,
        va="top",
        ha="center",
        color=dark_color
    )

# Formatting
ymin = min(np.min(data) for data in hourly_data) - 1
ymax = max(np.max(data) for data in hourly_data) + 1
ax.set_ylim(ymin, ymax)
ax.set_xticks(range(1, len(hourly_data) + 1))
ax.set_xticklabels(xlabels)
ax.tick_params(axis='x', which='minor', bottom=False, top=False)
ax.set_xlabel("EPC score band")
ax.set_ylabel(r"Centered outdoor temperature, $T_{in}$ ($^\circ$C)")

ax.set_title(
    "Centered outdoor"f" {room_name} ""temperature distributions by EPC score band\n"
    "EFUS 2017 Southwest subset, May-August"
)

ax.grid(axis="y")

plt.tight_layout()
plt.savefig(f"plots/efus2017/southwest_bedroom_4month/epc_toutc_distribution_{room_name}.svg", bbox_inches="tight")
plt.show()


In [ ]:
# EPC daily maximum indoor temperature violin plot

import matplotlib.colors as mcolors

# EPC labels
EPC_AXIS_LABELS = {
    1: "C+\n(SAP $>70$)",
    2: "D\n(SAP 51–70)",
    3: "E\n(SAP 30–50)",
    4: "F/G\n(SAP $<30$)",
}

display_bands = [b for b in [4, 3, 2, 1] if (df_m["EPceeb12e_efus"] == b).any()]

# Daily max indoor temperature
df_epc = df_m[[
    "CaseID",
    "hour",
    "T_in",
    "EPceeb12e_efus"
]].copy()

df_epc["date"] = df_epc["hour"].dt.floor("D")

daily_max = (
    df_epc
    .groupby(["CaseID", "date"])
    .agg(
        daily_max_T_in=("T_in", "max"),
        epc=("EPceeb12e_efus", "first")
    )
    .reset_index()
)

# EPC grouped data
_all_data    = [daily_max[daily_max["epc"] == b]["daily_max_T_in"].values for b in display_bands]
_all_labels  = [EPC_AXIS_LABELS[b] for b in display_bands]
_mask        = [len(d) > 0 for d in _all_data]
hourly_data  = [d for d, m in zip(_all_data, _mask) if m]
xlabels      = [l for l, m in zip(_all_labels, _mask) if m]

# Plot
fig, ax = plt.subplots(figsize=(3.4, 2.5))

parts = ax.violinplot(
    hourly_data,
    positions=range(1, len(hourly_data) + 1),
    showmeans=False,
    showmedians=False,
    showextrema=True
)

# Style violins
dark_colors = []
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

for i, pc in enumerate(parts["bodies"]):

    color = colors[i % len(colors)]
    dark_color = tuple(np.array(mcolors.to_rgb(color)) * 0.45)

    dark_colors.append(dark_color)

    pc.set_facecolor(color)
    pc.set_edgecolor(color)
    pc.set_alpha(0.45)

# Keep only central spine
parts["cmins"].set_visible(False)
parts["cmaxes"].set_visible(False)
parts["cbars"].set_linewidth(1.2)
parts["cbars"].set_color(dark_colors)

# Add IQR box, median line, and labels
for i, data in enumerate(hourly_data, start=1):

    color = colors[i - 1]
    dark_color = dark_colors[i - 1]

    q1, med, q3 = np.percentile(data, [25, 50, 75])

    ax.add_patch(
        plt.Rectangle(
            (i - 0.06, q1),
            0.12,
            q3 - q1,
            facecolor=color,
            edgecolor=dark_color,
            linewidth=1.2,
            zorder=4
        )
    )

    # Median line
    ax.plot(
        [i - 0.049, i + 0.049],
        [med, med],
        color="white",
        linewidth=2.2,
        solid_capstyle="butt",
        zorder=5
    )

    # Median label
    ax.text(
        i + 0.23,
        med,
        f"{med:.1f}°C",
        fontsize=5,
        va="center",
        ha="center",
        color=dark_color
    )

    # IQR label
    x_offset = 0.4 if i < len(hourly_data) else -0.4
    y_offset = q1 - 0.5 if i < len(hourly_data) else q3 + 0.5

    ax.text(
        i + x_offset,
        y_offset,
        f"IQR\n[{q1:.1f}, {q3:.1f}] °C",
        fontsize=5,
        va="top",
        ha="center",
        color=dark_color
    )

# Formatting
ymin = min(np.min(data) for data in hourly_data) - 1
ymax = max(np.max(data) for data in hourly_data) + 1

ax.set_ylim(ymin, ymax)
ax.set_xticks(range(1, len(hourly_data) + 1))
ax.set_xticklabels(xlabels)

ax.tick_params(axis='x', which='minor', bottom=False, top=False)

ax.set_xlabel("EPC score band")

ax.set_ylabel(
    "Daily maximum indoor temperature\n"
    r"$T_{in,max}$ ($^\circ$C)"
)

ax.set_title(
    f"Daily maximum indoor {room_name} temperature distributions by EPC score band\n"
    "EFUS 2017 Southwest subset, May-August"
)

ax.grid(axis="y")

plt.tight_layout()

plt.savefig(
    f"plots/efus2017/southwest_bedroom_4month/"
    f"epc_daily_max_tin_distribution_{room_name}.svg",
    bbox_inches="tight"
)

plt.show()

# Statistical comparison of daily maximum indoor temperatures
# Compare C+ against all other EPC bands

from scipy.stats import mannwhitneyu, ks_2samp, levene

print("\nStatistical comparison against EPC C+\n")

# Reference group: C+
ref_data = daily_max[
    daily_max["epc"] == 1
]["daily_max_T_in"].values

for b in [2, 3, 4]:

    comp_data = daily_max[
        daily_max["epc"] == b
    ]["daily_max_T_in"].values

    if len(comp_data) == 0 or len(ref_data) == 0:
        continue

    band_name = EPC_AXIS_LABELS[b].split("\n")[0]

    # ---------------------------------------------------------------------
    # 1. Mann–Whitney U test (difference in distributions / medians)
    # ---------------------------------------------------------------------

    mw_stat, mw_p = mannwhitneyu(
        ref_data,
        comp_data,
        alternative="greater"
    )

    # ---------------------------------------------------------------------
    # 2. Kolmogorov–Smirnov test (distribution shift)
    # ---------------------------------------------------------------------

    ks_stat, ks_p = ks_2samp(
        ref_data,
        comp_data,
        alternative="greater"
    )

    # ---------------------------------------------------------------------
    # 3. Levene test (difference in variability / spread)
    # ---------------------------------------------------------------------

    lev_stat, lev_p = levene(
        ref_data,
        comp_data,
        center="median"
    )

    print(f"\nC+ vs {band_name}")
    print("-" * 40)

    print(
        f"Mann–Whitney U : "
        f"U = {mw_stat:.1f}, p = {mw_p:.4g}"
    )

    print(
        f"Kolmogorov–Smirnov : "
        f"D = {ks_stat:.3f}, p = {ks_p:.4g}"
    )

    print(
        f"Levene (IQR/spread) : "
        f"W = {lev_stat:.3f}, p = {lev_p:.4g}"
    )


Daily maximum indoor temperatures were significantly higher in EPC C+ dwellings compared with D, E, and F/G bands (Mann–Whitney p<0.001), although differences in overall distribution shape (KS test) and variability (Levene test) were not statistically significant.

In [ ]:
# 2DMMT by EPC

import matplotlib.colors as mcolors

# EPC labels
EPC_AXIS_LABELS = {
    1: "C+\n(SAP $>70$)",
    2: "D\n(SAP 51–70)",
    3: "E\n(SAP 30–50)",
    4: "F/G\n(SAP $<30$)",
}

display_bands = [b for b in [4, 3, 2, 1] if (df_m["EPceeb12e_efus"] == b).any()]

# 2DMMT calculation
df_2dmmt = df_m[[
    "CaseID",
    "hour",
    "T_out",
    "EPceeb12e_efus"
]].copy()

# Daily maximum outdoor temperature
df_2dmmt["date"] = df_2dmmt["hour"].dt.floor("D")

daily_max = (
    df_2dmmt
    .groupby(["CaseID", "date"])
    .agg(
        daily_max_T_out=("T_out", "max"),
        epc=("EPceeb12e_efus", "first")
    )
    .reset_index()
)

# 2-day running mean of daily maxima
daily_max = daily_max.sort_values(["CaseID", "date"])

daily_max["T_2DMMT"] = (
    daily_max
    .groupby("CaseID")["daily_max_T_out"]
    .transform(lambda x: x.rolling(2, min_periods=1).mean())
)

# EPC grouped data
_all_data    = [daily_max[daily_max["epc"] == b]["T_2DMMT"].values for b in display_bands]
_all_labels  = [EPC_AXIS_LABELS[b] for b in display_bands]
_mask        = [len(d) > 0 for d in _all_data]
hourly_data  = [d for d, m in zip(_all_data, _mask) if m]
xlabels      = [l for l, m in zip(_all_labels, _mask) if m]

# Plot
fig, ax = plt.subplots(figsize=(3.4, 2.5))

parts = ax.violinplot(
    hourly_data,
    positions=range(1, len(hourly_data) + 1),
    showmeans=False,
    showmedians=False,
    showextrema=True
)

# Style violins
dark_colors = []
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

for i, pc in enumerate(parts["bodies"]):

    color = colors[i % len(colors)]
    dark_color = tuple(np.array(mcolors.to_rgb(color)) * 0.45)

    dark_colors.append(dark_color)

    pc.set_facecolor(color)
    pc.set_edgecolor(color)
    pc.set_alpha(0.45)

# Keep only central spine
parts["cmins"].set_visible(False)
parts["cmaxes"].set_visible(False)
parts["cbars"].set_linewidth(1.2)
parts["cbars"].set_color(dark_colors)

# Add IQR box, median line, and labels
for i, data in enumerate(hourly_data, start=1):

    color = colors[i - 1]
    dark_color = dark_colors[i - 1]

    q1, med, q3 = np.percentile(data, [25, 50, 75])

    ax.add_patch(
        plt.Rectangle(
            (i - 0.06, q1),
            0.12,
            q3 - q1,
            facecolor=color,
            edgecolor=dark_color,
            linewidth=1.2,
            zorder=4
        )
    )

    ax.plot(
        [i - 0.049, i + 0.049],
        [med, med],
        color="white",
        linewidth=2.2,
        solid_capstyle="butt",
        zorder=5
    )

    # Median label
    ax.text(
        i + 0.23,
        med,
        f"{med:.1f}°C",
        fontsize=5,
        va="center",
        ha="center",
        color=dark_color
    )

    # IQR label
    x_offset = 0.4 if i < len(hourly_data) else -0.4
    y_offset = q1 - 0.5 if i < len(hourly_data) else q3 + 0.5

    ax.text(
        i + x_offset,
        y_offset,
        f"IQR\n[{q1:.1f}, {q3:.1f}] °C",
        fontsize=5,
        va="top",
        ha="center",
        color=dark_color
    )

# Formatting
ymin = min(np.min(data) for data in hourly_data) - 1
ymax = max(np.max(data) for data in hourly_data) + 1

ax.set_ylim(ymin, ymax)
ax.set_xticks(range(1, len(hourly_data) + 1))
ax.set_xticklabels(xlabels)

ax.tick_params(axis="x", which="minor", bottom=False, top=False)

ax.set_xlabel("EPC score band")
ax.set_ylabel(r"Two-day Mean Max Temperature (2DMMT) ($^\circ$C)")

ax.set_title(
    f"Indoor {room_name} 2DMMT distributions by EPC score band\n"
    f"EFUS 2017 Southwest subset, May-August"
)

ax.grid(axis="y")

plt.tight_layout()

plt.savefig(
    f"plots/efus2017/southwest_bedroom_4month/epc_2dmmt_distribution_{room_name}.svg",
    bbox_inches="tight"
)

plt.show()


In [ ]:
# 2DMMT by AnyCooling

import matplotlib.colors as mcolors

# Cooling labels
COOLING_AXIS_LABELS = {
    0: "No\nCooling",
    1: "Cooling"
}

display_groups = [0, 1]

# 2DMMT calculation
df_2dmmt = df_m[[
    "CaseID",
    "hour",
    "T_out",
    "AnyCooling"
]].copy()

# Daily maximum outdoor temperature
df_2dmmt["date"] = df_2dmmt["hour"].dt.floor("D")

daily_max = (
    df_2dmmt
    .groupby(["CaseID", "date"])
    .agg(
        daily_max_T_out=("T_out", "max"),
        cooling=("AnyCooling", "first")
    )
    .reset_index()
)

# 2-day running mean of daily maxima
daily_max = daily_max.sort_values(
    ["CaseID", "date"]
)

daily_max["T_2DMMT"] = (
    daily_max
    .groupby("CaseID")["daily_max_T_out"]
    .transform(
        lambda x: x.rolling(
            2,
            min_periods=1
        ).mean()
    )
)

# Cooling grouped data
hourly_data = [
    daily_max[
        daily_max["cooling"] == g
    ]["T_2DMMT"].values
    for g in display_groups
]

xlabels = [
    COOLING_AXIS_LABELS[g]
    for g in display_groups
]

# Plot
fig, ax = plt.subplots(
    figsize=(2.6, 2.5)
)

parts = ax.violinplot(
    hourly_data,
    positions=range(
        1,
        len(display_groups) + 1
    ),
    showmeans=False,
    showmedians=False,
    showextrema=True
)

# Style violins
dark_colors = []

colors = plt.rcParams[
    "axes.prop_cycle"
].by_key()["color"]

for i, pc in enumerate(parts["bodies"]):

    color = colors[i % len(colors)]

    dark_color = tuple(
        np.array(
            mcolors.to_rgb(color)
        ) * 0.45
    )

    dark_colors.append(dark_color)

    pc.set_facecolor(color)
    pc.set_edgecolor(color)
    pc.set_alpha(0.45)

# Keep only central spine
parts["cmins"].set_visible(False)
parts["cmaxes"].set_visible(False)

parts["cbars"].set_linewidth(1.2)
parts["cbars"].set_color(dark_colors)

# Add IQR box, median line, and labels
for i, data in enumerate(hourly_data, start=1):

    color = colors[i - 1]
    dark_color = dark_colors[i - 1]

    q1, med, q3 = np.percentile(
        data,
        [25, 50, 75]
    )

    ax.add_patch(
        plt.Rectangle(
            (i - 0.06, q1),
            0.12,
            q3 - q1,
            facecolor=color,
            edgecolor=dark_color,
            linewidth=1.2,
            zorder=4
        )
    )

    ax.plot(
        [i - 0.049, i + 0.049],
        [med, med],
        color="white",
        linewidth=2.2,
        solid_capstyle="butt",
        zorder=5
    )

    # Median label
    ax.text(
        i + 0.23,
        med,
        f"{med:.1f}°C",
        fontsize=5,
        va="center",
        ha="center",
        color=dark_color
    )

    # IQR label
    x_offset = 0.38 if i == 1 else -0.38
    y_offset = q1 - 0.5 if i == 1 else q3 + 0.5

    ax.text(
        i + x_offset,
        y_offset,
        f"IQR\n[{q1:.1f}, {q3:.1f}] °C",
        fontsize=5,
        va="top",
        ha="center",
        color=dark_color
    )

# Formatting
ymin = min(np.min(data) for data in hourly_data) - 1
ymax = max(np.max(data) for data in hourly_data) + 1

ax.set_ylim(ymin, ymax)

ax.set_xticks(
    range(
        1,
        len(display_groups) + 1
    )
)

ax.set_xticklabels(xlabels)

ax.tick_params(
    axis="x",
    which="minor",
    bottom=False,
    top=False
)

ax.set_xlabel("Cooling presence")

ax.set_ylabel(
    r"Two-day Mean Max Temperature (2DMMT) ($^\circ$C)"
)

ax.set_title(
    f"Indoor {room_name} 2DMMT distributions by cooling presence\n"
    f"EFUS 2017 Southwest subset, May-August"
)

ax.grid(axis="y")

plt.tight_layout()

plt.savefig(
    f"plots/efus2017/southwest_bedroom_4month/"
    f"cooling_2dmmt_distribution_{room_name}.svg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# 2DMMnT by EPC

import matplotlib.colors as mcolors

# EPC labels
EPC_AXIS_LABELS = {
    1: "C+\n(SAP $>70$)",
    2: "D\n(SAP 51–70)",
    3: "E\n(SAP 30–50)",
    4: "F/G\n(SAP $<30$)",
}

display_bands = [b for b in [4, 3, 2, 1] if (df_m["EPceeb12e_efus"] == b).any()]

# Calculate 2-day mean minimum temperature (2DMMnT) per dwelling
df_2dmmnt = df_m[[
    "CaseID",
    "hour",
    "T_out",
    "EPceeb12e_efus"
]].copy()

df_2dmmnt["date"] = df_2dmmnt["hour"].dt.floor("D")

daily_min = (
    df_2dmmnt
    .groupby(["CaseID", "date"])
    .agg(
        daily_min_T_out=("T_out", "min"),
        epc=("EPceeb12e_efus", "first")
    )
    .reset_index()
)

# 2-day running mean of daily minima
daily_min = daily_min.sort_values(["CaseID", "date"])

daily_min["T_2DMMnT"] = (
    daily_min
    .groupby("CaseID")["daily_min_T_out"]
    .transform(lambda x: x.rolling(2, min_periods=1).mean())
)

# EPC grouped data
_all_data    = [daily_min[daily_min["epc"] == b]["T_2DMMnT"].values for b in display_bands]
_all_labels  = [EPC_AXIS_LABELS[b] for b in display_bands]
_mask        = [len(d) > 0 for d in _all_data]
hourly_data  = [d for d, m in zip(_all_data, _mask) if m]
xlabels      = [l for l, m in zip(_all_labels, _mask) if m]

# Plot
fig, ax = plt.subplots(figsize=(3.4, 2.5))

parts = ax.violinplot(
    hourly_data,
    positions=range(1, len(hourly_data) + 1),
    showmeans=False,
    showmedians=False,
    showextrema=True
)

# Style violins
dark_colors = []
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

for i, pc in enumerate(parts["bodies"]):

    color = colors[i % len(colors)]
    dark_color = tuple(np.array(mcolors.to_rgb(color)) * 0.45)

    dark_colors.append(dark_color)

    pc.set_facecolor(color)
    pc.set_edgecolor(color)
    pc.set_alpha(0.45)

# Keep only central spine
parts["cmins"].set_visible(False)
parts["cmaxes"].set_visible(False)
parts["cbars"].set_linewidth(1.2)
parts["cbars"].set_color(dark_colors)

# Add IQR box, median line, and labels
for i, data in enumerate(hourly_data, start=1):

    color = colors[i - 1]
    dark_color = dark_colors[i - 1]

    q1, med, q3 = np.percentile(data, [25, 50, 75])

    ax.add_patch(
        plt.Rectangle(
            (i - 0.06, q1),
            0.12,
            q3 - q1,
            facecolor=color,
            edgecolor=dark_color,
            linewidth=1.2,
            zorder=4
        )
    )

    ax.plot(
        [i - 0.049, i + 0.049],
        [med, med],
        color="white",
        linewidth=2.2,
        solid_capstyle="butt",
        zorder=5
    )

    # Median label
    ax.text(
        i + 0.23,
        med,
        f"{med:.1f}°C",
        fontsize=5,
        va="center",
        ha="center",
        color=dark_color
    )

    # IQR label
    x_offset = 0.4 if i < len(hourly_data) else -0.4
    y_offset = q1 - 0.5 if i < len(hourly_data) else q3 + 0.5

    ax.text(
        i + x_offset,
        y_offset,
        f"IQR\n[{q1:.1f}, {q3:.1f}] °C",
        fontsize=5,
        va="top",
        ha="center",
        color=dark_color
    )

# Formatting
ymin = min(np.min(data) for data in hourly_data) - 1
ymax = max(np.max(data) for data in hourly_data) + 1

ax.set_ylim(ymin, ymax)
ax.set_xticks(range(1, len(hourly_data) + 1))
ax.set_xticklabels(xlabels)

ax.tick_params(axis="x", which="minor", bottom=False, top=False)

ax.set_xlabel("EPC score band")
ax.set_ylabel("Two-day Mean Min Temperature\n(2DMMnT) ($^\\circ$C)")

ax.set_title(
    f"Indoor {room_name} 2DMMnT distributions by EPC score band\n"
    f"EFUS 2017 Southwest subset, May-August"
)

ax.grid(axis="y")

plt.tight_layout()

plt.savefig(
    f"plots/efus2017/southwest_bedroom_4month/epc_in_2dmmnt_distribution_{room_name}.svg",
    bbox_inches="tight"
)

plt.show()


In [ ]:
# 2DMMT + 2DMMnT by EPC

import matplotlib.colors as mcolors

# EPC labels
EPC_AXIS_LABELS = {
    1: "C+\n(SAP $>70$)",
    2: "D\n(SAP 51–70)",
    3: "E\n(SAP 30–50)",
    4: "F/G\n(SAP $<30$)",
}

display_bands = [b for b in [4, 3, 2, 1] if (df_m["EPceeb12e_efus"] == b).any()]

# Prepare dataset
df_temp = df_m[[
    "CaseID",
    "hour",
    "T_out",
    "EPceeb12e_efus"
]].copy()

df_temp["date"] = df_temp["hour"].dt.floor("D")

# Daily max/min outdoor temperatures
daily_stats = (
    df_temp
    .groupby(["CaseID", "date"])
    .agg(
        daily_max_T_out=("T_out", "max"),
        daily_min_T_out=("T_out", "min"),
        epc=("EPceeb12e_efus", "first")
    )
    .reset_index()
    .sort_values(["CaseID", "date"])
)

# Rolling metrics
daily_stats["T_2DMMT"] = (
    daily_stats
    .groupby("CaseID")["daily_max_T_out"]
    .transform(lambda x: x.rolling(2, min_periods=1).mean())
)

daily_stats["T_2DMMnT"] = (
    daily_stats
    .groupby("CaseID")["daily_min_T_out"]
    .transform(lambda x: x.rolling(2, min_periods=1).mean())
)

# Shared style
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

# Plotting helper
def epc_violin_plot(ax, metric, ylabel, title):

    _all_data    = [daily_stats[daily_stats["epc"] == b][metric].values for b in display_bands]
    _all_labels  = [EPC_AXIS_LABELS[b] for b in display_bands]
    _mask        = [len(d) > 0 for d in _all_data]
    hourly_data  = [d for d, m in zip(_all_data, _mask) if m]
    xlabels      = [l for l, m in zip(_all_labels, _mask) if m]

    if not hourly_data:
        return
    parts = ax.violinplot(
        hourly_data,
        positions=range(1, len(hourly_data) + 1),
        showmeans=False,
        showmedians=False,
        showextrema=True
    )

    dark_colors = []

    # Style violins
    for i, pc in enumerate(parts["bodies"]):

        color = colors[i % len(colors)]
        dark_color = tuple(np.array(mcolors.to_rgb(color)) * 0.45)

        dark_colors.append(dark_color)

        pc.set_facecolor(color)
        pc.set_edgecolor(color)
        pc.set_alpha(0.45)

    # Central spine only
    parts["cmins"].set_visible(False)
    parts["cmaxes"].set_visible(False)
    parts["cbars"].set_linewidth(1.2)
    parts["cbars"].set_color(dark_colors)

    # IQR + median
    for i, data in enumerate(hourly_data, start=1):

        color = colors[i - 1]
        dark_color = dark_colors[i - 1]

        q1, med, q3 = np.percentile(data, [25, 50, 75])

        ax.add_patch(
            plt.Rectangle(
                (i - 0.06, q1),
                0.12,
                q3 - q1,
                facecolor=color,
                edgecolor=dark_color,
                linewidth=1.2,
                zorder=4
            )
        )

        ax.plot(
            [i - 0.049, i + 0.049],
            [med, med],
            color="white",
            linewidth=2.2,
            solid_capstyle="butt",
            zorder=5
        )

        # Median label
        ax.text(
            i + 0.23,
            med,
            f"{med:.1f}°C",
            fontsize=5,
            va="center",
            ha="center",
            color=dark_color
        )

        # IQR label
        x_offset = 0.4 if i < len(hourly_data) else -0.4
        y_offset = q1 - 0.5 if i < len(hourly_data) else q3 + 0.5

        ax.text(
            i + x_offset,
            y_offset,
            f"IQR\n[{q1:.1f}, {q3:.1f}] °C",
            fontsize=5,
            va="top",
            ha="center",
            color=dark_color
        )

    # Formatting
    ymin = min(np.min(data) for data in hourly_data) - 1
    ymax = max(np.max(data) for data in hourly_data) + 1

    ax.set_ylim(ymin, ymax)
    ax.set_xticks(range(1, len(hourly_data) + 1))
    ax.set_xticklabels(xlabels)
    ax.tick_params(axis="x", which="minor", bottom=False, top=False)

    ax.set_xlabel("EPC score band")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y")

# 2DMMT
fig, ax = plt.subplots(figsize=(3.4, 2.5))

epc_violin_plot(
    ax=ax,
    metric="T_2DMMT",
    ylabel="Two-day Mean Max Temperature\n(2DMMT) ($^\\circ$C)",
    title=(
        f"Indoor {room_name} 2DMMT distributions by EPC score band\n"
        f"EFUS 2017 Southwest subset, May-August"
    )
)

plt.tight_layout()
plt.savefig(
    f"plots/efus2017/southwest_bedroom_4month/epc_2dmmt_distribution_{room_name}.svg",
    bbox_inches="tight"
)
plt.show()

# 2DMMnt
fig, ax = plt.subplots(figsize=(3.4, 2.5))

epc_violin_plot(
    ax=ax,
    metric="T_2DMMnT",
    ylabel="Two-day Mean Min Temperature\n(2DMMnT) ($^\\circ$C)",
    title=(
        f"Indoor {room_name} 2DMMnT distributions by EPC score band\n"
        f"EFUS 2017 Southwest subset, May-August"
    )
)

plt.tight_layout()

plt.savefig(
    f"plots/efus2017/southwest_bedroom_4month/epc_2dmmnt_distribution_{room_name}.svg",
    bbox_inches="tight"
)

plt.show()


In [ ]:
# EPC violin plots: T_in, T_out, T_out_c, 2DMMT, 2DMMnT

import matplotlib.colors as mcolors

# EPC labels
# EPC_AXIS_LABELS = {
#     1: "C+\n(SAP $>70$)",
#     2: "D\n(SAP 51–70)",
#     3: "E\n(SAP 30–50)",
#     4: "F/G\n(SAP $<30$)",
# }

EPC_AXIS_LABELS = {
    1: "C+",
    2: "D",
    3: "E",
    4: "F/G",
}

display_bands = [b for b in [4, 3, 2, 1] if (df_m["EPceeb12e_efus"] == b).any()]
xlabels = [EPC_AXIS_LABELS[b] for b in display_bands]

# Shared plotting style
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

# Base building characteristics 
df_epc = df_m[[
    "CaseID",
    "hour",
    "T_in",
    "T_out",
    "T_out_c",
    "EPceeb12e_efus"
]].copy()

# 2DMMT and 2DMMnT computation
df_epc["date"] = df_epc["hour"].dt.floor("D")

daily_stats = (
    df_epc
    .groupby(["CaseID", "date"])
    .agg(
        daily_max_T_out=("T_out", "max"),
        daily_min_T_out=("T_out", "min"),
        epc=("EPceeb12e_efus", "first")
    )
    .reset_index()
    .sort_values(["CaseID", "date"])
)

daily_stats["T_2DMMT"] = (
    daily_stats
    .groupby("CaseID")["daily_max_T_out"]
    .transform(lambda x: x.rolling(2, min_periods=1).mean())
)

daily_stats["T_2DMMnT"] = (
    daily_stats
    .groupby("CaseID")["daily_min_T_out"]
    .transform(lambda x: x.rolling(2, min_periods=1).mean())
)

# Violin plot function
def epc_violin_plot(data_groups, ylabel, title, save_path):
    
    # LaTeX article size 
    fig, ax = plt.subplots(figsize=(3.4, 2.5))

    parts = ax.violinplot(
        data_groups,
        positions=range(1, len(data_groups) + 1),
        showmeans=False,
        showmedians=False,
        showextrema=True
    )

    dark_colors = []

    # Style violins
    for i, pc in enumerate(parts["bodies"]):

        color = colors[i % len(colors)]
        dark_color = tuple(np.array(mcolors.to_rgb(color)) * 0.45)

        dark_colors.append(dark_color)

        pc.set_facecolor(color)
        pc.set_edgecolor(color)
        pc.set_alpha(0.45)

    # Central spine only
    parts["cmins"].set_visible(False)
    parts["cmaxes"].set_visible(False)
    parts["cbars"].set_linewidth(1.2)
    parts["cbars"].set_color(dark_colors)

    # IQR box + labels
    for i, data in enumerate(data_groups, start=1):

        color = colors[i - 1]
        dark_color = dark_colors[i - 1]

        q1, med, q3 = np.percentile(data, [25, 50, 75])

        ax.add_patch(
            plt.Rectangle(
                (i - 0.06, q1),
                0.12,
                q3 - q1,
                facecolor=color,
                edgecolor=dark_color,
                linewidth=1.2,
                zorder=4
            )
        )

        # Median line
        ax.plot(
            [i - 0.049, i + 0.049],
            [med, med],
            color="white",
            linewidth=2.2,
            solid_capstyle="butt",
            zorder=5
        )

        # Median label
        ax.text(
            i + 0.23,
            med-0.02,
            f"{med:.1f}°C",
            fontsize=5,
            va="center",
            ha="center",
            color=dark_color
        )

        # IQR label
        x_offset = 0.4 if i < len(data_groups) else -0.4
        y_offset = q1 - 0.5 if i < len(data_groups) else q3 + 0.5

        ax.text(
            i + x_offset,
            y_offset,
            f"IQR\n[{q1:.1f}, {q3:.1f}] °C",
            fontsize=5,
            va="top",
            ha="center",
            color=dark_color
        )

    # Formatting
    if not data_groups:
        return
    data_groups = [d for d in data_groups if len(d) > 0]
    if not data_groups:
        return
    ymin = min(np.min(data) for data in data_groups) - 1
    ymax = max(np.max(data) for data in data_groups) + 1

    ax.set_ylim(ymin, ymax)
    ax.set_xticks(range(1, len(data_groups) + 1))
    ax.set_xticklabels(xlabels)
    ax.tick_params(axis="x", which="minor", bottom=False, top=False)
    ax.set_xlabel("EPC score band")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y")

    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight")
    plt.show()

# Internal T
_all_data    = [df_epc[df_epc["EPceeb12e_efus"] == b]["T_in"].values for b in display_bands]
_all_labels  = [EPC_AXIS_LABELS[b] for b in display_bands]
_mask        = [len(d) > 0 for d in _all_data]
hourly_data  = [d for d, m in zip(_all_data, _mask) if m]
xlabels      = [l for l, m in zip(_all_labels, _mask) if m]

epc_violin_plot(
    data_groups=hourly_data,
    ylabel=r"Indoor temperature, $T_{in}$ ($^\circ$C)",
    title=(
        f"Indoor {room_name} temperature distributions by EPC score band\n"
        "EFUS 2017 Southwest subset, May-August"
    ),
    save_path=f"plots/efus2017/southwest_bedroom_4month/epc_tin_distribution_{room_name}.svg"
)

# External T
_all_data    = [df_epc[df_epc["EPceeb12e_efus"] == b]["T_out"].values for b in display_bands]
_mask        = [len(d) > 0 for d in _all_data]
hourly_data  = [d for d, m in zip(_all_data, _mask) if m]
epc_violin_plot(
    data_groups=hourly_data,
    ylabel=r"Outdoor temperature, $T_{out}$ ($^\circ$C)",
    title=(
        f"Outdoor temperature distributions by EPC score band\n"
        "EFUS 2017 Southwest subset, May-August"
    ),
    save_path=f"plots/efus2017/southwest_bedroom_4month/epc_tout_distribution_{room_name}.svg"
)

# External T (centered)
_all_data    = [df_epc[df_epc["EPceeb12e_efus"] == b]["T_out_c"].values for b in display_bands]
_mask        = [len(d) > 0 for d in _all_data]
hourly_data  = [d for d, m in zip(_all_data, _mask) if m]
epc_violin_plot(
    data_groups=hourly_data,
    ylabel=r"Centered outdoor temperature, $T_{out,c}$ ($^\circ$C)",
    title=(
        f"Centered outdoor temperature distributions by EPC score band\n"
        "EFUS 2017 Southwest subset, May-August"
    ),
    save_path=f"plots/efus2017/southwest_bedroom_4month/epc_toutc_distribution_{room_name}.svg"
)

# 2DMMT
_all_data    = [daily_stats[daily_stats["epc"] == b]["T_2DMMT"].values for b in display_bands]
_mask        = [len(d) > 0 for d in _all_data]
hourly_data  = [d for d, m in zip(_all_data, _mask) if m]
epc_violin_plot(
    data_groups=hourly_data,
    ylabel="Two-day Mean Max Temperature\n(2DMMT) ($^\\circ$C)",
    title=(
        f"2DMMT distributions by EPC score band\n"
        "EFUS 2017 Southwest subset, May-August"
    ),
    save_path=f"plots/efus2017/southwest_bedroom_4month/epc_2dmmt_distribution_{room_name}.svg"
)

# Outdoor 2DMMnT
_all_data    = [daily_stats[daily_stats["epc"] == b]["T_2DMMnT"].values for b in display_bands]
_mask        = [len(d) > 0 for d in _all_data]
hourly_data  = [d for d, m in zip(_all_data, _mask) if m]
epc_violin_plot(
    data_groups=hourly_data,
    ylabel="Outdoor Two-day Mean Min Temperature\n(2DMMnT) ($^\\circ$C)",
    title=(
        f"Outdoor 2DMMnT distributions by EPC score band\n"
        "EFUS 2017 Southwest subset, May-August"
    ),
    save_path=f"plots/efus2017/southwest_bedroom_4month/epc_out_2dmmnt_distribution_{room_name}.svg"
)


In [ ]:
""" Exceedance for the living room
In TM59, for the living room, overheating is for 3% of occupied hours (07:00–22:00) above threshold temperature
This meansa 15 occupied hours/day for hours 7–21 inclusive.


"""
# Occupied hours only
df_occ = df_m[
    df_m["hour"].dt.hour.between(7, 21)
].copy()

# Add date + 2DMMT to occupied-hour dataframe for exceedance plotting

df_occ["date"] = df_occ["hour"].dt.floor("D")

df_occ = df_occ.drop(columns=["T_2DMMT"], errors="ignore")

df_occ = df_occ.merge(
    daily_stats[["CaseID", "date", "T_2DMMT"]],
    on=["CaseID", "date"],
    how="left"
)

# Thresholds to test
thresholds = [26, 27, 28, 29]

# Exceedance frequency
results = []

for threshold in thresholds:

    exceed = (
        df_occ["T_in"] >= threshold
    ).astype(int)

    tmp = df_occ.copy()
    tmp["exceed"] = exceed

    # Frequency of exceedance per dwelling
    freq = (
        tmp.groupby("CaseID")
        .agg(
            exceed_hours=("exceed", "sum"),
            occupied_hours=("exceed", "count")
        )
        .reset_index()
    )

    freq["exceedance_pct"] = (
        100 * freq["exceed_hours"] / freq["occupied_hours"]
    )

    # TM59 overheating criterion (>3%)
    freq["overheating"] = freq["exceedance_pct"] > 3

    # Summary statistics
    n_overheat = freq["overheating"].sum()
    n_total = len(freq)

    results.append({
        "threshold": threshold,
        "n_overheat": n_overheat,
        "n_total": n_total,
        "pct_overheat": 100 * n_overheat / n_total,
        "median_exceedance": freq["exceedance_pct"].median(),
        "iqr_low": freq["exceedance_pct"].quantile(0.25),
        "iqr_high": freq["exceedance_pct"].quantile(0.75)
    })

results_df = pd.DataFrame(results)

print("\nTM59 living room exceedance analysis\n")

for _, row in results_df.iterrows():

    print(
        f"{row['threshold']:.0f}°C : "
        f"{row['n_overheat']:.0f}/{row['n_total']:.0f} dwellings "
        f"({row['pct_overheat']:.1f}%) exceeded TM59 criterion | "
        f"Median exceedance = {row['median_exceedance']:.1f}% "
        f"[IQR {row['iqr_low']:.1f}–{row['iqr_high']:.1f}%]"
    )

In [ ]:
# TM59 exceedance vs 2DMMT by EPC band
# Assumes df_occ, thresholds, and T_2DMMT already exist from previous cells
import matplotlib.ticker as mtick

fig, ax = plt.subplots(figsize=(3.4, 2.6))
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

for epc in [1, 2, 3, 4]:

    df_epc = df_occ[df_occ["EPceeb12e_efus"] == epc]
    epc_color = colors[4 - epc]

    for i, threshold in enumerate(thresholds):

        tmp = df_epc.copy()

        # Threshold exceedance
        tmp["exceed"] = (
            tmp["T_in"] >= threshold
        ).astype(int)

        # Dwelling-day exceedance
        freq = (
            tmp.groupby(["CaseID", "date", "T_2DMMT"])
            .agg(
                exceed_hours=("exceed", "sum"),
                occupied_hours=("exceed", "count")
            )
            .reset_index()
        )

        freq["exceedance_pct"] = (
            100 * freq["exceed_hours"] / freq["occupied_hours"]
        )

        freq["overheating"] = (
            freq["exceedance_pct"] > 3
        )

        # Rounded 2DMMT bins
        freq["T_2DMMT_bin"] = (
            freq["T_2DMMT"].round()
        )

        prop = (
            freq.groupby("T_2DMMT_bin")["overheating"]
            .mean()
            .reset_index()
        )

        ax.plot(
            prop["T_2DMMT_bin"],
            100 * prop["overheating"],
            marker="o",
            markersize=2.5,
            linewidth=1.2,
            linestyle=["-", "--", ":", "-."][i],
            color=epc_color,
            label=(
                f"{EPC_AXIS_LABELS[epc].split(chr(10))[0]} "
                f"| {threshold}°C"
            )
        )

# Formatting
ax.set_xlabel("Two-day Mean Max Temperature (2DMMT) ($^\\circ$C)")
ax.set_ylabel(
    "Living rooms exceeding TM59 criterion"
)
ax.set_ylim(0, 100)
ax.set_title(
    f"Indoor {room_name} TM59 exceedance by EPC score band and 2DMMT by EPC score band and 2DMMT\n"
        "EFUS 2017 Southwest subset, May-August"
)

ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.grid(axis="y")
ax.legend(
    fontsize=4.5,
    ncol=2,
    frameon=True
)

plt.tight_layout()

plt.savefig(
    f"plots/efus2017/southwest_bedroom_1week/"
    f"tm59_exceedance_vs_2dmmt_{room_name}.svg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# TM59 exceedance vs 2DMMT
# Separate subplot for each temperature threshold

import matplotlib.ticker as mtick

fig, axes = plt.subplots(
    2, 2,
    figsize=(6.8, 5),
    sharex=True,
    sharey=True
)

axes = axes.flatten()

colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

for threshold, ax in zip(thresholds, axes):

    for epc in [1, 2, 3, 4]:

        df_epc = df_occ[df_occ["EPceeb12e_efus"] == epc]
        epc_color = colors[4 - epc]

        tmp = df_epc.copy()

        # Threshold exceedance
        tmp["exceed"] = (
            tmp["T_in"] >= threshold
        ).astype(int)

        # Dwelling-day exceedance
        freq = (
            tmp.groupby(["CaseID", "date", "T_2DMMT"])
            .agg(
                exceed_hours=("exceed", "sum"),
                occupied_hours=("exceed", "count")
            )
            .reset_index()
        )

        freq["exceedance_pct"] = (
            100 * freq["exceed_hours"] / freq["occupied_hours"]
        )

        freq["overheating"] = (
            freq["exceedance_pct"] > 3
        )

        # Rounded 2DMMT bins
        freq["T_2DMMT_bin"] = (
            freq["T_2DMMT"].round()
        )

        prop = (
            freq.groupby("T_2DMMT_bin")["overheating"]
            .mean()
            .reset_index()
        )

        ax.plot(
            prop["T_2DMMT_bin"],
            100 * prop["overheating"],
            marker="o",
            markersize=2.5,
            linewidth=1.4,
            color=epc_color,
            label=EPC_AXIS_LABELS[epc].split(chr(10))[0]
        )

    # Subplot formatting
    ax.set_title(
        f"{threshold}$^\\circ$C threshold",
        fontsize=8
    )

    ax.set_xlabel(
    "Two-day Mean Max Temperature (2DMMT) ($^\\circ$C)")

    ax.tick_params(axis="x", labelbottom=True)
    
    ax.set_ylim(0, 100)

    ax.yaxis.set_major_formatter(
        mtick.PercentFormatter()
    )

    ax.grid(axis="y")

# Shared labels
# fig.supxlabel(
#     "Two-day Mean Max Temperature (2DMMT) ($^\\circ$C)"
# )

fig.supylabel(
    "Living rooms exceeding TM59 criterion"
)

# Shared legend
handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=4,
    frameon=True,
    fontsize=6,
    title="EPC band",
    bbox_to_anchor=(0.5, -0.05)
)

fig.suptitle(
    f"Indoor {room_name} TM59 exceedance by EPC score band and 2DMMT\n"
    "EFUS 2017 Southwest subset, May-August",
    y=1.03
)

plt.tight_layout()

plt.savefig(
    f"plots/efus2017/southwest_bedroom_4month/"
    f"tm59_exceedance_vs_2dmmt_threshold_subplots_{room_name}.svg",
    bbox_inches="tight"
)

plt.show()

# Cumulative TM59 exceedance vs 2DMMT
# Separate subplot for each temperature threshold

import matplotlib.ticker as mtick

fig, axes = plt.subplots(
    2, 2,
    figsize=(6.8, 5),
    sharex=True,
    sharey=True
)

axes = axes.flatten()

colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

for threshold, ax in zip(thresholds, axes):

    for epc in [1, 2, 3, 4]:

        df_epc = df_occ[df_occ["EPceeb12e_efus"] == epc]
        epc_color = colors[4 - epc]

        tmp = df_epc.copy()

        # Threshold exceedance
        tmp["exceed"] = (
            tmp["T_in"] >= threshold
        ).astype(int)

        # Dwelling-day exceedance
        freq = (
            tmp.groupby(["CaseID", "date", "T_2DMMT"])
            .agg(
                exceed_hours=("exceed", "sum"),
                occupied_hours=("exceed", "count")
            )
            .reset_index()
        )

        freq["exceedance_pct"] = (
            100 * freq["exceed_hours"] / freq["occupied_hours"]
        )

        freq["overheating"] = (
            freq["exceedance_pct"] > 3
        )

        # Sort by 2DMMT
        freq = freq.sort_values("T_2DMMT")

        # Cumulative overheating frequency
        freq["cum_overheating"] = (
            100 * freq["overheating"].cumsum()
            / np.arange(1, len(freq) + 1)
        )

        ax.plot(
            freq["T_2DMMT"],
            freq["cum_overheating"],
            marker="o",
            markersize=0,
            linewidth=1.4,
            color=epc_color,
            label=EPC_AXIS_LABELS[epc].split(chr(10))[0]
        )

    ax.set_yscale('log')

    # Subplot formatting
    ax.set_title(
        f"{threshold}$^\\circ$C threshold",
        fontsize=8
    )

    ax.set_xlabel(
        "Two-day Mean Max Temperature (2DMMT) ($^\\circ$C)"
    )

    ax.tick_params(axis="x", labelbottom=True)

    ax.set_ylim(0, 100)

    ax.yaxis.set_major_formatter(
        mtick.PercentFormatter()
    )

    ax.grid(axis="y")

# Shared labels
fig.supylabel(
    "Cumulative living rooms exceeding TM59 criterion"
)

# Shared legend
handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=4,
    frameon=True,
    fontsize=6,
    title="EPC band",
    bbox_to_anchor=(0.5, -0.05)
)

fig.suptitle(
    f"Indoor {room_name} cumulative TM59 exceedance by EPC score band and 2DMMT\n"
    "EFUS 2017 Southwest subset, May-August",
    y=1.03
)

plt.tight_layout()

plt.savefig(
    f"plots/efus2017/southwest_bedroom_4month/"
    f"tm59_cumulative_exceedance_vs_2dmmt_threshold_subplots_{room_name}.svg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Distribution of exceedance percentages by EPC band
# Styled consistently with previous violin plots

import matplotlib.colors as mcolors

thresholds = [26,27,28]
# thresholds = [26, 27, 28]

colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

fig, axes = plt.subplots(
    1, len(thresholds),
    figsize=(6.8, 2.6),
    sharey=True
)

for threshold, ax in zip(thresholds, axes):

    exceedance_results = []

    for epc in display_bands:

        tmp = df_occ[df_occ["EPceeb12e_efus"] == epc].copy()

        tmp["exceed"] = (
            tmp["T_in"] >= threshold
        ).astype(int)

        freq = (
            tmp.groupby("CaseID")
            .agg(
                exceed_hours=("exceed", "sum"),
                occupied_hours=("exceed", "count")
            )
            .reset_index()
        )

        freq["exceedance_pct"] = (
            100 * freq["exceed_hours"] / freq["occupied_hours"]
        )

        exceedance_results.append(
            freq["exceedance_pct"].values
        )

    # Violin plot
    parts = ax.violinplot(
        exceedance_results,
        positions=range(1, len(hourly_data) + 1),
        showmeans=False,
        showmedians=False,
        showextrema=True
    )

    dark_colors = []

    # Style violins
    for i, pc in enumerate(parts["bodies"]):

        color = colors[i]
        dark_color = tuple(np.array(mcolors.to_rgb(color)) * 0.45)

        dark_colors.append(dark_color)

        pc.set_facecolor(color)
        pc.set_edgecolor(color)
        pc.set_alpha(0.45)

    # Keep only central spine
    parts["cmins"].set_visible(False)
    parts["cmaxes"].set_visible(False)

    parts["cbars"].set_linewidth(1.2)
    parts["cbars"].set_color(dark_colors)

    # IQR + median styling
    for i, data in enumerate(exceedance_results, start=1):

        color = colors[i - 1]
        dark_color = dark_colors[i - 1]

        q1, med, q3 = np.percentile(data, [25, 50, 75])

        # IQR box
        ax.add_patch(
            plt.Rectangle(
                (i - 0.06, q1),
                0.12,
                q3 - q1,
                facecolor=color,
                edgecolor=dark_color,
                linewidth=1.2,
                zorder=4
            )
        )

        # Median line
        ax.plot(
            [i - 0.044, i + 0.044],
            [med, med],
            color="white",
            linewidth=1.2,
            solid_capstyle="butt",
            zorder=5
        )

        # Median label
        ax.text(
            i + 0.23,
            med,
            f"{med:.1f}%",
            fontsize=5,
            va="center",
            ha="center",
            color=dark_color
        )

       # IQR label at top of violin spine
        y_offset = np.max(data) + 2

        ax.text(
            i,
            y_offset,
            f"IQR\n[{q1:.1f}, {q3:.1f}]",
            fontsize=5,
            va="bottom",
            ha="center",
            color=dark_color
        )

    # Formatting
    # ymin = min(np.min(d) for d in exceedance_results)
    # ymax = max(np.max(d) for d in exceedance_results)

    ax.set_ylim([0, 50])

    ax.set_xticks(range(1, len(hourly_data) + 1))

    ax.set_xticklabels([
        EPC_AXIS_LABELS[b] for b in display_bands
    ])

    ax.tick_params(
        axis="x",
        which="minor",
        bottom=False,
        top=False
    )

    ax.set_title(
        f"{threshold}$^\\circ$C threshold",
        fontsize=8
    )

    ax.grid(axis="y")

    ax.yaxis.set_major_formatter(
        mtick.PercentFormatter()
    )

axes[0].set_ylabel(
    "Fraction of occupied hours above threshold"
)

fig.supxlabel(
    "EPC score band"
)

fig.suptitle(
    f"Indoor {room_name} exceedance distributions by EPC band\n"
    "EFUS 2017 Southwest subset, May-August",
    y=1.03
)

plt.tight_layout()

plt.savefig(
    f"plots/efus2017/southwest_bedroom_4month/"
    f"tm59_exceedance_distribution_{room_name}.svg",
    bbox_inches="tight"
)

plt.show()


In [ ]:
# Probability of overheating by EPC band
# Styled consistently with previous plots

import matplotlib.ticker as mtick
import matplotlib.colors as mcolors

fig, axes = plt.subplots(
    1, len(thresholds),
    figsize=(6.8, 2.6),
    sharey=True
)

for threshold, ax in zip(thresholds, axes):

    probs = []
    dark_colors = []

    overheating_counts = []
    total_counts = []

    for epc in display_bands:

        tmp = df_occ[df_occ["EPceeb12e_efus"] == epc].copy()

        tmp["exceed"] = (
            tmp["T_in"] >= threshold
        ).astype(int)

        freq = (
            tmp.groupby("CaseID")
            .agg(
                exceed_hours=("exceed", "sum"),
                occupied_hours=("exceed", "count")
            )
            .reset_index()
        )

        freq["exceedance_pct"] = (
            100 * freq["exceed_hours"] / freq["occupied_hours"]
        )

        overheating = (
            freq["exceedance_pct"] > 3
        )

        overheating_prob = overheating.mean()

        probs.append(100 * overheating_prob)

        overheating_counts.append(
            overheating.sum()
        )

        total_counts.append(
            len(overheating)
        )

        color = colors[len(probs) - 1]

        dark_colors.append(
            tuple(np.array(mcolors.to_rgb(color)) * 0.45)
        )

    # Bars
    bars = ax.bar(
        range(1, len(display_bands) + 1),
        probs,
        color=colors[:len(display_bands)],
        width=0.65,
        alpha=0.45,
        linewidth=1.2
    )

    # Bar styling + labels
    for i, (bar, prob) in enumerate(zip(bars, probs)):

        bar.set_edgecolor(dark_colors[i])

        n_overheat = overheating_counts[i]
        n_total = total_counts[i]

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            prob + 2,
            # f"{prob:.1f}%\n({n_overheat}/{n_total})",
            f"({n_overheat}/{n_total})",
            fontsize=5,
            ha="center",
            va="bottom",
            color=dark_colors[i]
        )

    # Formatting
    ax.set_title(
        f"{threshold}$^\\circ$C threshold",
        fontsize=8
    )

    ax.set_xticks(range(1, len(display_bands) + 1))

    ax.set_xticklabels([
        EPC_AXIS_LABELS[b]
        for b in display_bands
    ])

    ax.set_ylim(0, 100)

    ax.grid(axis="y")

    ax.yaxis.set_major_formatter(
        mtick.PercentFormatter()
    )

axes[0].set_ylabel(
    "Fraction of dwellings classified as overheating"
)

fig.supxlabel(
    "EPC score band"
)

fig.suptitle(
    f"Indoor {room_name} overheating probability by EPC band\n"
    r"TM59 criterion ($>3\%$ occupied hours)",
    y=1.03
)

plt.tight_layout()

plt.savefig(
    f"plots/efus2017/southwest_bedroom_4month/"
    f"tm59_overheating_probability_{room_name}.svg",
    bbox_inches="tight"
)

plt.show()


In [ ]:
# Degree-hours exceedance
# Styled consistently with previous violin plots

import matplotlib.colors as mcolors

fig, axes = plt.subplots(
    1, len(thresholds),
    figsize=(6.8, 2.6),
    sharey=True
)

for threshold, ax in zip(thresholds, axes):

    dh_results = []

    for epc in display_bands:

        tmp = df_occ[df_occ["EPceeb12e_efus"] == epc].copy()

        # Degree-hours above threshold
        tmp["degree_hours"] = (
            tmp["T_in"] - threshold
        ).clip(lower=0)

        dh = (
            tmp.groupby("CaseID")
            .agg(
                degree_hours=("degree_hours", "sum")
            )
            .reset_index()
        )

        dh_results.append(
            dh["degree_hours"].values
        )

    # Violin plot
    parts = ax.violinplot(
        dh_results,
        positions=range(1, len(hourly_data) + 1),
        showmeans=False,
        showmedians=False,
        showextrema=True
    )

    dark_colors = []

    # Style violins
    for i, pc in enumerate(parts["bodies"]):

        color = colors[i]
        dark_color = tuple(np.array(mcolors.to_rgb(color)) * 0.45)

        dark_colors.append(dark_color)

        pc.set_facecolor(color)
        pc.set_edgecolor(color)
        pc.set_alpha(0.45)

    # Central spine only
    parts["cmins"].set_visible(False)
    parts["cmaxes"].set_visible(False)

    parts["cbars"].set_linewidth(1.2)
    parts["cbars"].set_color(dark_colors)

    # IQR + median styling
    for i, data in enumerate(dh_results, start=1):

        color = colors[i - 1]
        dark_color = dark_colors[i - 1]

        q1, med, q3 = np.percentile(data, [25, 50, 75])

        # IQR box
        ax.add_patch(
            plt.Rectangle(
                (i - 0.06, q1),
                0.12,
                q3 - q1,
                facecolor=color,
                edgecolor=dark_color,
                linewidth=1.2,
                zorder=4
            )
        )

        # Median line
        ax.plot(
            [i - 0.049, i + 0.049],
            [med, med],
            color="white",
            linewidth=1.2,
            solid_capstyle="butt",
            zorder=5
        )

        # Median label
        ax.text(
            i + 0.23,
            med,
            f"{med:.1f}",
            fontsize=5,
            va="center",
            ha="center",
            color=dark_color
        )

        # IQR label at top of violin spine
        y_offset = np.max(data) + 2

        ax.text(
            i,
            y_offset,
            f"IQR\n[{q1:.1f}, {q3:.1f}]",
            fontsize=5,
            va="bottom",
            ha="center",
            color=dark_color
        )

    # Formatting
    ymin = min(np.min(d) for d in dh_results)
    ymax = max(np.max(d) for d in dh_results)

    ax.set_ylim([
        0,400
    ])

    ax.set_xticks(range(1, len(hourly_data) + 1))

    ax.set_xticklabels([
        EPC_AXIS_LABELS[b]
        for b in display_bands
    ])

    ax.tick_params(
        axis="x",
        which="minor",
        bottom=False,
        top=False
    )

    ax.set_title(
        f"{threshold}$^\\circ$C threshold",
        fontsize=8
    )

    ax.grid(axis="y")

axes[0].set_ylabel(
    "Degree-hours above threshold"
)

fig.supxlabel(
    "EPC score band"
)

fig.suptitle(
    f"Indoor {room_name} overheating severity by EPC band\n"
    "Degree-hours exceedance",
    y=1.03
)

plt.tight_layout()

plt.savefig(
    f"plots/efus2017/southwest_bedroom_4month/"
    f"tm59_degree_hours_{room_name}.svg",
    bbox_inches="tight"
)

plt.show()


In [ ]:
# Duration analysis: longest continuous overheating event
# Styled consistently with previous violin plots

import matplotlib.colors as mcolors

fig, axes = plt.subplots(
    1, len(thresholds),
    figsize=(6.8, 2.6),
    sharey=True
)

# Store results for printing later
all_duration_results = {}

for threshold, ax in zip(thresholds, axes):

    duration_results = []

    # Create nested dict for this threshold
    all_duration_results[threshold] = {}

    for epc in display_bands:

        tmp = (
            df_occ[df_occ["EPceeb12e_efus"] == epc]
            .sort_values(["CaseID", "hour"])
            .copy()
        )

        tmp["exceed"] = (
            tmp["T_in"] >= threshold
        ).astype(int)

        durations = []

        # Store CaseID -> longest duration
        duration_by_house = {}

        for case_id, group in tmp.groupby("CaseID"):

            runs = (
                group["exceed"]
                .groupby(
                    (
                        group["exceed"]
                        != group["exceed"].shift()
                    ).cumsum()
                )
                .cumsum()
            )

            max_duration = runs.max()

            durations.append(max_duration)

            duration_by_house[case_id] = max_duration

        # Save for later printing
        all_duration_results[threshold][epc] = duration_by_house

        duration_results.append(
            np.array(durations)
        )

    # Violin plot
    parts = ax.violinplot(
        duration_results,
        positions=range(1, len(hourly_data) + 1),
        showmeans=False,
        showmedians=False,
        showextrema=True
    )

    dark_colors = []

    # Style violins
    for i, pc in enumerate(parts["bodies"]):

        color = colors[i]

        dark_color = tuple(
            np.array(mcolors.to_rgb(color)) * 0.45
        )

        dark_colors.append(dark_color)

        pc.set_facecolor(color)
        pc.set_edgecolor(color)
        pc.set_alpha(0.45)

    # Keep only central spine
    parts["cmins"].set_visible(False)
    parts["cmaxes"].set_visible(False)

    parts["cbars"].set_linewidth(1.2)
    parts["cbars"].set_color(dark_colors)

    # IQR + median styling
    for i, data in enumerate(duration_results, start=1):

        color = colors[i - 1]
        dark_color = dark_colors[i - 1]

        q1, med, q3 = np.percentile(
            data,
            [25, 50, 75]
        )

        # IQR box
        ax.add_patch(
            plt.Rectangle(
                (i - 0.06, q1),
                0.12,
                q3 - q1,
                facecolor=color,
                edgecolor=dark_color,
                linewidth=1.2,
                zorder=4
            )
        )

        # Median line
        ax.plot(
            [i - 0.049, i + 0.049],
            [med, med],
            color="white",
            linewidth=2.2,
            solid_capstyle="butt",
            zorder=5
        )

        # Median label
        ax.text(
            i + 0.23,
            med,
            f"{med:.1f} h",
            fontsize=5,
            va="center",
            ha="center",
            color=dark_color
        )

        # IQR label
        y_offset = np.max(data) + 1

        ax.text(
            i,
            y_offset,
            f"IQR\n[{q1:.1f}, {q3:.1f}]",
            fontsize=5,
            va="bottom",
            ha="center",
            color=dark_color
        )

    # Formatting
    ax.set_ylim([
        0, 300
    ])

    ax.set_xticks(range(1, len(hourly_data) + 1))

    ax.set_xticklabels([
        EPC_AXIS_LABELS[b]
        for b in display_bands
    ])

    ax.tick_params(
        axis="x",
        which="minor",
        bottom=False,
        top=False
    )

    ax.set_title(
        f"{threshold}$^\\circ$C threshold",
        fontsize=8
    )

    ax.grid(axis="y")

axes[0].set_ylabel(
    "Longest continuous overheating event (hours)"
)

fig.supxlabel(
    "EPC score band"
)

fig.suptitle(
    f"Indoor {room_name} overheating duration by EPC band\n"
    "Longest continuous overheating event",
    y=1.03
)

plt.tight_layout()

plt.savefig(
    f"plots/efus2017/southwest_bedroom_4month/"
    f"tm59_duration_{room_name}.svg",
    bbox_inches="tight"
)

plt.show()

print("\nLongest overheating events by EPC band")

for threshold in thresholds:

    print("\n" + "=" * 60)
    print(f"Threshold: {threshold}°C")
    print("=" * 60)

    for epc in display_bands:

        duration_by_house = all_duration_results[threshold][epc]

        sorted_houses = sorted(
            duration_by_house.items(),
            key=lambda x: x[1],
            reverse=True
        )

        print(f"\nEPC {epc}")
        print("-" * 30)

        for case_id, duration in sorted_houses:

            print(
                f"CaseID {case_id}: "
                f"{duration:.0f} hours"
            )


In [ ]:
plot_temps(list(df_m["dwelling"].unique()[:4]))

In [ ]:
# Get all post-1990 dwellings
post_1990_ids = (
    dwelling_chars[
        dwelling_chars["dwage_efus"] == 7
    ]["CaseID"]
    .astype(str)
    .unique()
    .tolist()
)

print(
    f"Number of post-1990 dwellings: "
    f"{len(post_1990_ids)}"
)

# Plot all post-1990 dwellings
plot_temps(post_1990_ids)

In [ ]:
# TM59 exceedance envelope vs 2DMMT by EPC band
# Filled region spans min–max exceedance across thresholds

import matplotlib.ticker as mtick

fig, ax = plt.subplots(figsize=(3.4, 2.6))

colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

for epc in display_bands:

    df_epc = df_occ[df_occ["EPceeb12e_efus"] == epc]
    epc_color = colors[4 - epc]

    threshold_props = []

    # Calculate exceedance curves for all thresholds
    for threshold in thresholds:

        tmp = df_epc.copy()

        tmp["exceed"] = (
            tmp["T_in"] >= threshold
        ).astype(int)

        freq = (
            tmp.groupby(["CaseID", "date", "T_2DMMT"])
            .agg(
                exceed_hours=("exceed", "sum"),
                occupied_hours=("exceed", "count")
            )
            .reset_index()
        )

        freq["exceedance_pct"] = (
            100 * freq["exceed_hours"] / freq["occupied_hours"]
        )

        freq["overheating"] = (
            freq["exceedance_pct"] > 3
        )

        freq["T_2DMMT_bin"] = (
            freq["T_2DMMT"].round()
        )

        prop = (
            freq.groupby("T_2DMMT_bin")["overheating"]
            .mean()
            .reset_index()
        )

        prop["overheating"] *= 100

        threshold_props.append(
            prop.set_index("T_2DMMT_bin")["overheating"]
        )

    # Combine thresholds into one dataframe
    prop_df = pd.concat(threshold_props, axis=1)

    x = prop_df.index.values
    y_min = prop_df.min(axis=1).values
    y_max = prop_df.max(axis=1).values
    y_mid = prop_df.mean(axis=1).values

    # Filled range
    ax.fill_between(
        x,
        y_min,
        y_max,
        color=epc_color,
        alpha=0.2,
        label=EPC_AXIS_LABELS[epc].split(chr(10))[0]
    )

    # Mean line
    ax.plot(
        x,
        y_mid,
        color=epc_color,
        linewidth=1.5
    )

# Formatting
ax.set_xlabel("Two-day Mean Max Temperature (2DMMT) ($^\\circ$C)")

ax.set_ylabel(
    "Living rooms exceeding TM59 criterion"
)

ax.set_ylim(0, 100)
ax.set_xlim(min(x),max(x))

ax.set_title(
    f"Indoor {room_name} TM59 exceedance by EPC score band and 2DMMT\n"
    "EFUS 2017 Southwest subset, May-August"
)

ax.yaxis.set_major_formatter(mtick.PercentFormatter())

ax.grid(axis="y")

ax.legend(
    fontsize=5,
    frameon=False,
    loc='upper left'
)

plt.tight_layout()

plt.savefig(
    f"plots/efus2017/southwest_bedroom_1week/"
    f"tm59_exceedance_vs_2dmmt_{room_name}.svg",
    bbox_inches="tight"
)

plt.show()

### Mixed Effects Models

#### Model 1: Baseline
$$\boxed{T_{\mathrm{in},ij} = \beta_0 + \beta_1\,\tilde{T}_{\mathrm{out},ij} + u_{0j} + u_{1j}\,\tilde{T}_{\mathrm{out},ij} + \varepsilon_{ij}}$$

#### Model 2: Population-averaged diurnal shape
$$\boxed{T_{\mathrm{in},ij} = \beta_0 + \beta_1\,\tilde{T}_{\mathrm{out},ij} + \beta_2\,s_{ij} + \beta_3\,c_{ij} + u_{0j} + u_{1j}\,\tilde{T}_{\mathrm{out},ij} + \varepsilon_{ij}}$$

#### Model 3: 23 hour-of-day fixed effects
$$\boxed{T_{\mathrm{in},ij} = \beta_0 + \beta_1\tilde{T}_{\mathrm{out},ij} + \sum_{h=1}^{23}\gamma_h\,\mathbf{1}[h_{ij} = h] + u_{0j}+u_{1j}\tilde{T}_{\mathrm{out},ij} + \varepsilon_{ij}}$$

**Notation**

| Symbol | Definition |
|:---|:---|
| $i,j$ | Observation $i$ in dwelling $j$ |
| $T_{\mathrm{in},ij}$ | Indoor temperature (°C) |
| $\tilde{T}_{\mathrm{out},ij} = T_{\mathrm{out},ij} - \bar{T}_{\mathrm{out}}$ | Centred outdoor temperature |
| $s_{ij} = \sin(2\pi h_{ij}/24)$,  $c_{ij} = \cos(2\pi h_{ij}/24)$ | Single-harmonic diurnal basis |
| $\gamma_h$ | Hour-of-day offset vs midnight ($h=0$, reference) |
| $u_{0j},\,u_{1j}$ | Dwelling random intercept and slope |
| $\varepsilon_{ij}$ | Within-dwelling residual |

**Random effects structure**

$$u_j = \begin{pmatrix} u_{0j} \\ u_{1j} \end{pmatrix} \sim \mathcal{N}(\mathbf{0},\,D), \qquad D = \begin{pmatrix} \sigma^2_{u0} & \sigma_{u01} \\ \sigma_{u01} & \sigma^2_{u1} \end{pmatrix}, \qquad \varepsilon_{ij} \sim \mathcal{N}(0,\sigma^2_\varepsilon)$$

All models fitted by maximum likelihood (ML) to allow AIC comparison across different fixed-effect structures.


In [ ]:
import statsmodels.formula.api as smf
import numpy as np
import warnings
warnings.filterwarnings("ignore")

RE = "~T_out_c"   # random intercept + random slope on T_out_c

def fit_model(formula, method="powell"):
    md = smf.mixedlm(formula, df_m, groups=df_m["dwelling"], re_formula=RE)
    return md.fit(reml=False, method=method, maxiter=2000)

def var_components(res):
    cov = res.cov_re
    return {
        "sig2_u0":  float(cov.iloc[0, 0]),
        "sig2_u1":  float(cov.iloc[1, 1]),
        "sig_u01":  float(cov.iloc[0, 1]),
        "sig2_eps": float(res.scale),
    }

# ── Fit models ────────────────────────────────────────────────────────────────
FORMULAS = {
    "Null": "T_in ~ 1",
    "M1":   "T_in ~ T_out_c",
    "M2":   "T_in ~ T_out_c + sin_h + cos_h",
    "M3":   "T_in ~ T_out_c + C(hour_of_day)",
}

results = {}
for name, formula in FORMULAS.items():
    results[name] = fit_model(formula)

null_vc    = var_components(results["Null"])
null_total = null_vc["sig2_u0"] + null_vc["sig2_u1"] + null_vc["sig2_eps"]

# ── Comparison table ──────────────────────────────────────────────────────────
W    = 12
COLS = ["Null", "M1", "M2", "M3"]
vcs  = {n: var_components(results[n]) for n in COLS}
tots = {n: vcs[n]["sig2_u0"] + vcs[n]["sig2_u1"] + vcs[n]["sig2_eps"] for n in COLS}
pcts = {n: 100 * (1 - tots[n] / null_total) for n in COLS}
ref  = results["M1"].aic

def cells(vals, fmt):
    return "".join(format(v, fmt) for v in vals)

print("=" * 74)
print("  M1–M3 COMPARISON")
print("=" * 74)
print("  " + " " * 26 + "".join(f"{'  ' + c:>{W}}" for c in COLS))
print("-" * 74)
print("  " + f"{'AIC':<26}"              + cells([results[n].aic         for n in COLS], f"{W}.1f"))
print("  " + f"{'ΔAIC vs M1':<26}"       + cells([results[n].aic - ref   for n in COLS], f"+{W}.1f"))
print("  " + f"{'% variance reduced':<26}" + "".join(f"{pcts[n]:{W}.1f}%" for n in COLS))
print("-" * 74)
for label, key in [("σ²_u0", "sig2_u0"), ("σ²_u1", "sig2_u1"),
                   ("σ_u01", "sig_u01"), ("σ²_ε",  "sig2_eps")]:
    print("  " + f"{label:<26}" + cells([vcs[n][key] for n in COLS], f"{W}.4f"))
print("  " + f"{'ΣVC':<26}"              + cells([tots[n]           for n in COLS], f"{W}.4f"))
print("=" * 74)

# ── Parameter tables ──────────────────────────────────────────────────────────
def pstars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""

def print_fe(name, res):
    vc   = var_components(res)
    fe   = res.fe_params
    se   = res.bse_fe
    pv   = res.pvalues
    corr = vc["sig_u01"] / (vc["sig2_u0"] * vc["sig2_u1"]) ** 0.5

    print(f"\n{'─' * 62}")
    print(f"  FIXED EFFECTS — {name}")
    print(f"{'─' * 62}")
    print(f"  {'Parameter':<28} {'Coef':>8}  {'SE':>7}  {'p':>9}")
    print(f"  {'-' * 57}")

    key_params  = [p for p in fe.index if not p.startswith("C(hour_of_day)")]
    hour_params = [p for p in fe.index if p.startswith("C(hour_of_day)")]

    for p in key_params:
        print(f"  {p:<28} {fe[p]:>8.3f}  {se[p]:>7.3f}  {pv[p]:>9.4f}  {pstars(pv[p])}")

    if hour_params:
        vals = fe[hour_params]
        ph   = int(vals.idxmax().split("[T.")[1].rstrip("]"))
        th   = int(vals.idxmin().split("[T.")[1].rstrip("]"))
        print(f"  {'C(hour_of_day) [23 dummies]':<28} "
              f"range [{vals.min():.3f}, {vals.max():+.3f}] °C  "
              f"peak {ph:02d}:00  trough {th:02d}:00")

    print(f"\n  Random:  σ_u0={vc['sig2_u0']**0.5:.4f}  "
          f"σ_u1={vc['sig2_u1']**0.5:.4f}  "
          f"corr(u0,u1)={corr:.3f}  "
          f"σ_ε={vc['sig2_eps']**0.5:.4f}")

for name in ["M1", "M2", "M3"]:
    print_fe(name, results[name])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.graphics.gofplots import qqplot
from statsmodels.tsa.stattools import acf
from statsmodels.stats.diagnostic import acorr_ljungbox

def plot_diagnostics(name, res, max_lag=24*7):

    resid  = res.resid
    fitted = res.fittedvalues

    print(f"\n{name}: computing diagnostics...")

    fig, axes = plt.subplots(2, 2, figsize=(6.8, 5))

    # ── Residuals vs fitted ──────────────────────────────────────────────────
    ax = axes[0, 0]

    ax.scatter(fitted, resid, s=4, alpha=0.3)

    ax.axhline(0, color="red", linestyle="--")

    ax.set_title(f"{name}: Residuals vs Fitted")
    ax.set_xlabel("Fitted")
    ax.set_ylabel("Residuals")

    # ── QQ plot ──────────────────────────────────────────────────────────────
    ax = axes[0, 1]

    qqplot(resid, line="45", ax=ax, fit=True)

    ax.set_title(f"{name}: Normal Q-Q")

    # ── ACF ──────────────────────────────────────────────────────────────────
    ax = axes[1, 0]

    print(f"{name}: computing ACF...")

    acf_vals, confint = acf(
        resid,
        nlags=max_lag,
        fft=True,
        alpha=0.05
    )

    lags = np.arange(len(acf_vals))

    ax.vlines(lags, 0, acf_vals)

    ax.axhline(0, color="black", linewidth=0.8)

    # 95% CI
    lower = confint[:, 0] - acf_vals
    upper = confint[:, 1] - acf_vals

    ax.fill_between(
        lags,
        lower,
        upper,
        alpha=0.2
    )

    # daily markers
    for lag in np.arange(0, max_lag + 1, 24):

        ax.axvline(
            lag,
            color='red',
            linestyle='--',
            alpha=0.4
        )

        ax.text(
            lag,
            ax.get_ylim()[1] * 0.95,
            f"Day {lag // 24}",
            color='red',
            rotation=90,
            verticalalignment='top',
            horizontalalignment='right',
            alpha=0.7
        )

    ax.set_title(f"{name}: Residual ACF")
    ax.set_xlabel("Lag (hours)")
    ax.set_ylabel("ACF")

    ax.set_ylim(-1, 1)

    ax = axes[1, 1]
    ax.hist(resid, bins=50)
    ax.set_title(f"{name}: Residual Histogram")
    ax.set_xlabel("Residual")
    ax.set_ylabel("Count")

    plt.tight_layout()

for name in ["M1", "M2", "M3"]:

    plot_diagnostics(name, results[name])

    plt.savefig(
        f"plots/efus2017/southwest_bedroom_4month/"
        f"mixedlm_diagnostics_{name}.svg",
        bbox_inches="tight"
    )
    plt.show()
    plt.close()

#### Model 4: M3 + binary building characteristics
$$\boxed{T_{\mathrm{in},ij} = \beta_0 + \beta_1\tilde{T}_{\mathrm{out},ij} + \sum_{h=1}^{23}\gamma_h\,\mathbf{1}[h_{ij}=h] + \boldsymbol{\delta}^\top\mathbf{b}_j + u_{0j}+u_{1j}\tilde{T}_{\mathrm{out},ij} + \varepsilon_{ij}}$$

#### Model 5: M4 + categorical building characteristics
$$\boxed{T_{\mathrm{in},ij} = \beta_0 + \beta_1\tilde{T}_{\mathrm{out},ij} + \sum_{h=1}^{23}\gamma_h\,\mathbf{1}[h_{ij}=h] + \boldsymbol{\delta}^\top\mathbf{b}_j + \boldsymbol{\alpha}^\top\mathbf{c}_j + u_{0j}+u_{1j}\tilde{T}_{\mathrm{out},ij} + \varepsilon_{ij}}$$

#### Model 6: M2 + binary building characteristics
$$\boxed{T_{\mathrm{in},ij} = \beta_0 + \beta_1\tilde{T}_{\mathrm{out},ij} + \beta_2 s_{ij} + \beta_3 c_{ij} + \boldsymbol{\delta}^\top\mathbf{b}_j + u_{0j}+u_{1j}\tilde{T}_{\mathrm{out},ij} + \varepsilon_{ij}}$$

#### Model 7: M6 + categorical building characteristics
$$\boxed{T_{\mathrm{in},ij} = \beta_0 + \beta_1\tilde{T}_{\mathrm{out},ij} + \beta_2 s_{ij} + \beta_3 c_{ij} + \boldsymbol{\delta}^\top\mathbf{b}_j + \boldsymbol{\alpha}^\top\mathbf{c}_j + u_{0j}+u_{1j}\tilde{T}_{\mathrm{out},ij} + \varepsilon_{ij}}$$

**Additional notation**

| Symbol | Definition |
|:---|:---|
| $\mathbf{b}_j = (B_{1j},B_{2j},B_{3j},B_{4j})^\top$ | Binary building characteristics for dwelling $j$ |
| $B_{1j}$ | CavityWall: 1 if cavity wall (WallType2x\_efus = 2), 0 if solid |
| $B_{2j}$ | InsulatedWalls\_efus: 1 if wall insulation present, 0 otherwise |
| $B_{3j}$ | FullyDblGlz\_efus: 1 if fully double-glazed, 0 otherwise |
| $B_{4j}$ | AnyCooling: 1 if any mechanical cooling, 0 otherwise |
| $\boldsymbol{\delta}$ | Coefficients for binary building characteristics |
| $\mathbf{c}_j$ | Treatment-coded categorical indicators: dwelling type, age, floor area, EPC band |
| $\boldsymbol{\alpha}$ | Coefficients for categorical indicators (contrast vs reference level) |

M4 and M5 use 23 hour-of-day dummies (M3 time term). M6 and M7 use the single harmonic $s_{ij},c_{ij}$ (M2 time term). All models share the same random-effects structure: random intercept and random slope on $\tilde{T}_{\mathrm{out}}$ grouped by dwelling, fitted by ML.

In [ ]:
import re

# ── Binary indicator: cavity wall (WallType2x_efus == 2) ─────────────────────
df_m["CavityWall"] = (df_m["WallType2x_efus"] == 2).astype(int)

# ── Model formulas ────────────────────────────────────────────────────────────
BIN = "CavityWall + InsulatedWalls_efus + FullyDblGlz_efus + AnyCooling"
CAT = "C(dwtype_efus) + C(dwage_efus) + C(floor6x_efus) + C(EPceeb12e_efus)"

NEW_FORMULAS = {
    "M4": f"T_in ~ T_out_c + C(hour_of_day) + {BIN}",
    "M5": f"T_in ~ T_out_c + C(hour_of_day) + {BIN} + {CAT}",
    "M6": f"T_in ~ T_out_c + sin_h + cos_h + {BIN}",
    "M7": f"T_in ~ T_out_c + sin_h + cos_h + {BIN} + {CAT}",
}

for name, formula in NEW_FORMULAS.items():
    print(f"Fitting {name} ...")
    results[name] = fit_model(formula)
    print(f"  {'converged' if results[name].converged else 'NOT CONVERGED'}  AIC = {results[name].aic:.1f}")

# ── Full comparison table: Null, M1–M7 ───────────────────────────────────────
ALL  = ["Null", "M1", "M2", "M3", "M4", "M5", "M6", "M7"]
vcs  = {n: var_components(results[n]) for n in ALL}
tots = {n: vcs[n]["sig2_u0"] + vcs[n]["sig2_u1"] + vcs[n]["sig2_eps"] for n in ALL}
pcts = {n: 100 * (1 - tots[n] / null_total) for n in ALL}
ref_aic = results["M3"].aic

# k = n_fe_params + 4 RE params (σ²_u0, σ²_u1, σ_u01, σ²_ε)
n_fe  = {n: len(results[n].fe_params) for n in ALL}
k_all = {n: n_fe[n] + 4 for n in ALL}

print(f"\n{'='*105}")
print("  FULL MODEL COMPARISON  (Null – M7)")
print(f"{'='*105}")
print(f"  {'Model':<6} {'log-lik':>12} {'AIC':>12} {'ΔAIC':>9} {'k':>4} {'%VarRed':>8} "
      f"{'σ²_u0':>8} {'σ²_u1':>7} {'σ²_ε':>8} {'ΣVC':>8}")
print(f"  {'-'*101}")
for n in ALL:
    daic = format(results[n].aic - ref_aic, "+9.2f")
    pstr = f"{pcts[n]:>7.1f}%"
    print(f"  {n:<6} {results[n].llf:>12.2f} {results[n].aic:>12.2f} {daic} "
          f"{k_all[n]:>4} {pstr} "
          f"{vcs[n]['sig2_u0']:>8.4f} {vcs[n]['sig2_u1']:>7.4f} "
          f"{vcs[n]['sig2_eps']:>8.4f} {tots[n]:>8.4f}")
print(f"  {'='*101}")
print(f"  ΔAIC relative to M3 (AIC = {ref_aic:.2f})")
print(f"  k = number of estimated parameters (fixed effects + 4 variance components)")

# ── Human-readable labels ─────────────────────────────────────────────────────
REF_CATS = {
    "dwtype_efus":    1,
    "dwage_efus":     1,
    "floor6x_efus":   1,
    "EPceeb12e_efus": 1,
}
BINARY_LABELS = {
    "CavityWall":          "Cavity wall: Yes vs Solid",
    "InsulatedWalls_efus": "Insulated walls: Yes vs No",
    "FullyDblGlz_efus":    "Full double glazing: Yes vs No",
    "AnyCooling":          "Any cooling: Yes vs No",
}

def human_label(param):
    m = re.match(r"C\((\w+)\)\[T\.(\d+)\]", param)
    if m:
        var, code = m.group(1), int(m.group(2))
        lev  = LEVELS.get(var, {}).get(code, str(code))
        rlev = LEVELS.get(var, {}).get(REF_CATS.get(var, 1), "ref")
        return f"{lev} vs {rlev}"
    return BINARY_LABELS.get(param, param)

# ── Parameter tables for M4–M7 ───────────────────────────────────────────────
def print_fe_bldg(name, res):
    fe   = res.fe_params
    se   = res.bse_fe
    pv   = res.pvalues
    vc   = var_components(res)
    corr = vc["sig_u01"] / (vc["sig2_u0"] * vc["sig2_u1"]) ** 0.5

    GROUPS = [
        ("Core",
            [p for p in fe.index if p in ("Intercept", "T_out_c", "sin_h", "cos_h")]),
        ("Hour of day",
            [p for p in fe.index if p.startswith("C(hour_of_day)")]),
        ("Binary characteristics",
            [p for p in fe.index if p in BINARY_LABELS]),
        ("Dwelling type",
            [p for p in fe.index if "dwtype"  in p]),
        ("Dwelling age",
            [p for p in fe.index if "dwage"   in p]),
        ("Floor area",
            [p for p in fe.index if "floor6x" in p]),
        ("EPC band",
            [p for p in fe.index if "EPceeb"  in p]),
    ]

    print(f"\n{'─'*68}")
    print(f"  FIXED EFFECTS — {name}")
    print(f"{'─'*68}")
    print(f"  {'Parameter':<44} {'Coef':>7}  {'SE':>7}  {'p':>9}")
    print(f"  {'-'*64}")

    for group, params in GROUPS:
        if not params:
            continue
        if group == "Hour of day":
            vals = fe[params]
            ph   = int(vals.idxmax().split("[T.")[1].rstrip("]"))
            th   = int(vals.idxmin().split("[T.")[1].rstrip("]"))
            print(f"\n  {group}")
            print(f"    {'23 dummies':<42} [{vals.min():.3f},{vals.max():+.3f}] °C  "
                  f"peak {ph:02d}:00  trough {th:02d}:00")
            continue
        ref_var = {
            "Dwelling type": "dwtype_efus",
            "Dwelling age":  "dwage_efus",
            "Floor area":    "floor6x_efus",
            "EPC band":      "EPceeb12e_efus",
        }.get(group)
        print(f"\n  {group}", end="")
        if ref_var:
            ref_label = LEVELS.get(ref_var, {}).get(REF_CATS.get(ref_var, 1), "")
            print(f"  (ref: {ref_label})", end="")
        print()
        for p in params:
            stars = "***" if pv[p]<0.001 else "**" if pv[p]<0.01 else "*" if pv[p]<0.05 else ""
            print(f"    {human_label(p):<44} {fe[p]:>7.3f}  {se[p]:>7.3f}  {pv[p]:>9.4f}  {stars}")

    print(f"\n  Random:  σ_u0={vc['sig2_u0']**0.5:.4f}  "
          f"σ_u1={vc['sig2_u1']**0.5:.4f}  "
          f"corr(u0,u1)={corr:.3f}  "
          f"σ_ε={vc['sig2_eps']**0.5:.4f}")

for name in ["M4", "M5", "M6", "M7"]:
    print_fe_bldg(name, results[name])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.graphics.gofplots import qqplot
from statsmodels.tsa.stattools import acf
from statsmodels.stats.diagnostic import acorr_ljungbox

def plot_diagnostics(name, res, max_lag=24*7):

    resid  = res.resid
    fitted = res.fittedvalues

    print(f"\n{name}: computing diagnostics...")

    fig, axes = plt.subplots(2, 2, figsize=(6.8, 5))

    # ── Residuals vs fitted ──────────────────────────────────────────────────
    ax = axes[0, 0]

    ax.scatter(fitted, resid, s=4, alpha=0.3)

    ax.axhline(0, color="red", linestyle="--")

    ax.set_title(f"{name}: Residuals vs Fitted")
    ax.set_xlabel("Fitted")
    ax.set_ylabel("Residuals")

    # ── QQ plot ──────────────────────────────────────────────────────────────
    ax = axes[0, 1]

    qqplot(resid, line="45", ax=ax, fit=True)

    ax.set_title(f"{name}: Normal Q-Q")

    # ── ACF ──────────────────────────────────────────────────────────────────
    ax = axes[1, 0]

    print(f"{name}: computing ACF...")

    acf_vals, confint = acf(
        resid,
        nlags=max_lag,
        fft=True,
        alpha=0.05
    )

    lags = np.arange(len(acf_vals))

    ax.vlines(lags, 0, acf_vals)

    ax.axhline(0, color="black", linewidth=0.8)

    # 95% CI
    lower = confint[:, 0] - acf_vals
    upper = confint[:, 1] - acf_vals

    ax.fill_between(
        lags,
        lower,
        upper,
        alpha=0.2
    )

    # daily markers
    for lag in np.arange(0, max_lag + 1, 24):

        ax.axvline(
            lag,
            color='red',
            linestyle='--',
            alpha=0.4
        )

        ax.text(
            lag,
            ax.get_ylim()[1] * 0.95,
            f"Day {lag // 24}",
            color='red',
            rotation=90,
            verticalalignment='top',
            horizontalalignment='right',
            alpha=0.7
        )

    ax.set_title(f"{name}: Residual ACF")
    ax.set_xlabel("Lag (hours)")
    ax.set_ylabel("ACF")

    ax.set_ylim(-1, 1)

    ax = axes[1, 1]
    ax.hist(resid, bins=50)
    ax.set_title(f"{name}: Residual Histogram")
    ax.set_xlabel("Residual")
    ax.set_ylabel("Count")

    plt.tight_layout()
    return fig

for name in ["M1", "M2", "M3", "M4", "M5", "M6", "M7"]:

    fig = plot_diagnostics(name, results[name])

    fig.savefig(
        f"plots/efus2017/southwest_bedroom_4month/"
        f"mixedlm_diagnostics_{name}.svg",
        bbox_inches="tight"
    )

    plt.show()

    plt.close(fig)

### Model Comparison Summary

All models fitted by maximum likelihood (ML, `reml=False`) on 69 London dwellings, May–August 2018 (196,492 obs). Random effects: random intercept + random slope on $\tilde{T}_{\mathrm{out}}$ by dwelling. Goodness-of-fit (% variance reduced) is $100 \times (1 - \Sigma\mathrm{VC}_m / \Sigma\mathrm{VC}_{\mathrm{null}})$ where $\Sigma\mathrm{VC} = \sigma^2_{u0} + \sigma^2_{u1} + \sigma^2_\varepsilon$. ΔAIC relative to M3.

#### Static models (full dataset, 196,492 obs)

| Model | Description | log-lik | AIC | ΔAIC | k | % Var Red | $\sigma^2_{u0}$ | $\sigma^2_{u1}$ | $\sigma^2_\varepsilon$ | $\beta_1$ |
|:------|:------------|--------:|----:|-----:|--:|----------:|----------------:|----------------:|------------------------:|-----------:|
| Null | Intercept only | −391,001 | 782,013 | +69,715 | 5 | 0.0% | 1.553 | 0.071 | 3.117 | — |
| M1 | + $\tilde{T}_{\mathrm{out}}$ | −390,918 | 781,848 | +69,551 | 6 | 1.2% | 1.559 | 0.006 | 3.117 | 0.256 |
| M2 | + sin/cos (harmonic) | −356,672 | 713,361 | +1,063 | 8 | 20.1% | 1.581 | 0.005 | 2.199 | 0.387 |
| **M3** | **+ 23 hour dummies** | **−356,120** | **712,298** | **0** | **29** | **20.4%** | **1.581** | **0.005** | **2.187** | **0.389** |
| M4 | M3 + binary building chars | −356,118 | 712,302 | +4.2 | 33 | 22.2% | 1.496 | 0.005 | 2.187 | 0.389 |
| **M5** | **M4 + categorical building chars** | **−356,095** | **712,294** | **−4.3** | **52** | **38.1%** | **0.742** | **0.005** | **2.187** | **0.389** |
| M6 | M2 + binary building chars | −356,671 | 713,365 | +1,067 | 12 | 21.9% | 1.496 | 0.005 | 2.199 | 0.387 |
| M7 | M6 + categorical building chars | −356,647 | 713,357 | +1,059 | 31 | 37.9% | 0.742 | 0.005 | 2.199 | 0.387 |

Binary building chars: CavityWall, InsulatedWalls\_efus, FullyDblGlz\_efus, AnyCooling. Categorical: dwelling type, age, floor area, EPC band. k = fixed effects + 4 variance components. **M5 wins by AIC** (ΔAIC = −4.3 vs M3).

#### Dynamic models (df\_ar: 194,805 consecutive-hour pairs)

Models M8–M10 add a lagged response $\phi\,T_{\mathrm{in},i-1,j}$ to the fixed-effects structure of M3/M5/M7. Static models are re-fitted on `df_ar` for a fair AIC comparison. ΔAIC is relative to the corresponding static re-fit.

| Model | Base | log-lik | AIC | ΔAIC vs static ref | $\phi$ | $\beta_{1,\mathrm{SR}}$ | $\beta_{1,\mathrm{LR}}$ | $\sigma_{u0}$ | $\sigma_{u1}$ | $\sigma_\varepsilon$ |
|:------|:-----|--------:|----:|-------------------:|------:|-------------------------:|-------------------------:|---------------:|---------------:|--------------------:|
| M3\_ar | re-fit M3 on df\_ar | −352,884 | 705,825 | (ref) | — | 0.389 | — | 1.258 | 0.073 | 1.479 |
| **M8** | **M3 + lag** | **−12,729** | **25,517** | **−680,308** | **0.9657** | **0.019** | **0.561** | **0.046** | **0.009** | **0.258** |
| M5\_ar | re-fit M5 on df\_ar | −352,858 | 705,821 | (ref) | — | 0.389 | — | 0.861 | 0.073 | 1.479 |
| **M9** | **M5 + lag** | **−12,704** | **25,515** | **−680,306** | **0.9656** | **0.019** | **0.560** | **0.032** | **0.009** | **0.258** |
| M7\_ar | re-fit M7 on df\_ar | −353,404 | 706,869 | (ref) | — | 0.387 | — | 0.861 | 0.073 | 1.483 |
| M10 | M7 + lag | −13,696 | 27,456 | −679,413 | 0.9645 | 0.020 | 0.565 | 0.033 | 0.009 | 0.259 |

$\beta_{1,\mathrm{LR}} = \beta_{1,\mathrm{SR}} / (1 - \phi)$ is the long-run outdoor sensitivity. **M9 wins by AIC** among dynamic models (AIC = 25,515).

---

#### Comments

**1. Random slope collapse (Null → M1).**  
The between-dwelling random slope variance $\sigma^2_{u1}$ drops from 0.071 to 0.006 as soon as $\tilde{T}_{\mathrm{out}}$ enters the fixed effects. The population-average outdoor sensitivity is absorbed by $\beta_1$, leaving almost no unexplained dwelling-to-dwelling variation in slope. From M1 onwards, the residual random structure is dominated by the random intercept $\sigma^2_{u0}$ (chronic baseline differences between dwellings).

**2. $\beta_1$ attenuation in M1.**  
$\beta_1 = 0.256$ °C/°C in M1 vs 0.387–0.389 in M2/M3. Without a time-of-day control, outdoor temperature is confounded with the diurnal cycle: $T_{\mathrm{out}}$ peaks around 14:00, the same hour when indoor temperatures are suppressed (ventilation, thermal mass). The cross-correlation biases the naïve slope downward by ~34%.

**3. Time-of-day specification (M2 vs M3).**  
The single harmonic (M2) reduces $\sigma^2_\varepsilon$ from 3.117 to 2.199 (−29%) and improves AIC by 68,488 units over M1. The 23 hour dummies (M3) add a further 1,063 AIC improvement — a substantially larger margin than in the 1-month August subset — because the 3-month window contains more diurnal fine structure (spring/summer transitions) that the harmonic cannot represent.

**4. Binary building characteristics (M4, M6).**  
None of the four binary predictors (cavity wall, wall insulation, full double glazing, any cooling) are statistically significant at 5%. M4 worsens AIC by +4.2 relative to M3; M6 worsens it by +1,067. $\sigma^2_{u0}$ falls by only 5% (1.581 → 1.496), confirming binary recoding has insufficient resolution at $n = 69$ dwellings.

**5. Categorical building characteristics (M5, M7).**  
Adding dwelling type, age, floor area and EPC band reduces $\sigma^2_{u0}$ by 53% (1.581 → 0.742). M5 beats M3 by ΔAIC = −4.3, reversing the conclusion from the 1-month August analysis (where M3 was optimal). M5 vs M4: ΔAIC = −8.5 (19 extra parameters pay off on the larger 3-month dataset). **M5 is the best static model by AIC.**

**6. Significant categorical contrasts (M5/M7).**  
- *Dwelling age*: post-war builds (1945–1964: +1.3 °C; 1965–1974: +1.6 °C vs pre-1919, both $p < 0.05$). Mid-century construction — cavity walls without modern insulation standards, larger glazing areas — retains solar gain more than Victorian stock.  
- *EPC band*: Bands D (−0.80 °C, $p = 0.049$), E (−1.10 °C, $p = 0.021$) and F/G (−1.50 °C, $p = 0.011$) are cooler than C+. In summer, high-efficiency (C+) dwellings are warmer: air-tight, well-insulated fabric that reduces heating bills in winter traps heat in May–August.  
- *Any cooling* (M5/M7): marginally significant (+0.59 °C, $p \approx 0.04$). Reverse causation: cooling is adopted in response to overheating, not the other way around.

**7. Best static model: M5.**  
M5 (hour dummies + all building characteristics) is the preferred static model by AIC (712,294). It reduces $\sigma^2_{u0}$ from 1.581 to 0.742 relative to M3, explaining building-fabric-driven baseline heterogeneity. M7 (sin/cos + building chars) is 1,063 AIC units worse than M5, confirming hour dummies outperform harmonics in the 3-month window.

**8. Dynamic models (M8–M10): thermal inertia dominates.**  
Adding a lagged response $\phi T_{\mathrm{in},i-1}$ produces ΔAIC improvements of ~680,000 units — the thermal AR structure is overwhelming. $\phi \approx 0.966$ across all three dynamic models: current indoor temperature is almost entirely determined by the previous hour. The short-run outdoor sensitivity collapses to $\beta_{1,\mathrm{SR}} \approx 0.019$ °C/°C, but the long-run effect $\beta_{1,\mathrm{LR}} \approx 0.56$ °C/°C is consistent with the static estimates. Between-dwelling variance $\sigma^2_{u0}$ collapses from ~1.58 to ~0.04: once the AR(1) state is modelled, most apparent chronic dwelling differences are explained by persistent AR history rather than fixed dwelling characteristics. **M9** (M5 + lag) is the best dynamic model by AIC (25,515).


## Dealing with Autocorrelation

Sinusoidal ACF reflects the thermal inertia of buildings. Model this as AR(1) errors:

$$\varepsilon_{ij} = \rho\,\varepsilon_{i-1,j} + a_{ij}, \quad a_{ij} \sim N(0,\sigma_a^2)$$

which implies a structured residual covariance:

$$
\mathrm{Var}(\boldsymbol{\varepsilon}_j) = \sigma^2 R(\rho), \qquad
R(\rho) = \begin{pmatrix}
1 & \rho & \rho^2 & \cdots \\
\rho & 1 & \rho & \cdots \\
\rho^2 & \rho & 1 & \cdots \\
\vdots & \vdots & \vdots & \ddots
\end{pmatrix}
$$

#### Model 8: Baseline
$$\boxed{T_{\mathrm{in},ij} = \beta_0 + \beta_1\,\tilde{T}_{\mathrm{out},ij} + u_{0j} + u_{1j}\,\tilde{T}_{\mathrm{out},ij} + \varepsilon_{ij}}$$

#### Model 9: Population-averaged diurnal shape
$$\boxed{T_{\mathrm{in},ij} = \beta_0 + \beta_1\,\tilde{T}_{\mathrm{out},ij} + \beta_2\,s_{ij} + \beta_3\,c_{ij} + u_{0j} + u_{1j}\,\tilde{T}_{\mathrm{out},ij} + \varepsilon_{ij}}$$

#### Model 10: 23 hour-of-day fixed effects
$$\boxed{T_{\mathrm{in},ij} = \beta_0 + \beta_1\tilde{T}_{\mathrm{out},ij} + \sum_{h=1}^{23}\gamma_h\,\mathbf{1}[h_{ij} = h] + u_{0j}+u_{1j}\tilde{T}_{\mathrm{out},ij} + \varepsilon_{ij}}$$


Implemented in `efus_mm_london_4month`



#### Model 8: M3 + lagged response (dynamic M3)

$$\boxed{T_{\mathrm{in},ij} = \phi\,T_{\mathrm{in},i-1,j} + \beta_0 + \beta_1\tilde{T}_{\mathrm{out},ij} + \sum_{h=1}^{23}\gamma_h\,\mathbf{1}[h_{ij}=h] + u_{0j}+u_{1j}\tilde{T}_{\mathrm{out},ij} + a_{ij}}$$

#### Model 9: M5 + lagged response (dynamic M5)

$$\boxed{T_{\mathrm{in},ij} = \phi\,T_{\mathrm{in},i-1,j} + \beta_0 + \beta_1\tilde{T}_{\mathrm{out},ij} + \sum_{h=1}^{23}\gamma_h\,\mathbf{1}[h_{ij}=h] + \boldsymbol{\delta}^\top\mathbf{b}_j + \boldsymbol{\alpha}^\top\mathbf{c}_j + u_{0j}+u_{1j}\tilde{T}_{\mathrm{out},ij} + a_{ij}}$$

#### Model 10: M7 + lagged response (dynamic M7)

$$\boxed{T_{\mathrm{in},ij} = \phi\,T_{\mathrm{in},i-1,j} + \beta_0 + \beta_1\tilde{T}_{\mathrm{out},ij} + \beta_2 s_{ij} + \beta_3 c_{ij} + \boldsymbol{\delta}^\top\mathbf{b}_j + \boldsymbol{\alpha}^\top\mathbf{c}_j + u_{0j}+u_{1j}\tilde{T}_{\mathrm{out},ij} + a_{ij}}$$

where $\mathbf{b}_j$ = binary building characteristics (CavityWall, InsulatedWalls, FullyDblGlz, AnyCooling) and $\mathbf{c}_j$ = categorical building characteristics (dwelling type, age, floor area, EPC band).

In [ ]:
# ── Lagged dataset: consecutive-hour pairs only ────────────────────
df_ar = df_m.sort_values(["dwelling", "hour"]).reset_index(drop=True)
df_ar["T_in_lag"]  = df_ar.groupby("dwelling")["T_in"].shift(1)
df_ar["_hour_lag"] = df_ar.groupby("dwelling")["hour"].shift(1)
df_ar["_dt_h"]     = (df_ar["hour"] - df_ar["_hour_lag"]).dt.total_seconds() / 3600
df_ar = df_ar[df_ar["_dt_h"] == 1.0].dropna(subset=["T_in_lag"]).reset_index(drop=True)
print(f"AR dataset: {len(df_ar):,} obs  ({len(df_m) - len(df_ar)} first-obs dropped)  "
      f"{df_ar['dwelling'].nunique()} dwellings\n")

# ── Fit helper on df_ar ────────────────────────────────────────
def fit_ar(formula, method="powell"):
    md = smf.mixedlm(formula, df_ar, groups=df_ar["dwelling"], re_formula="~T_out_c")
    return md.fit(reml=False, method=method, maxiter=2000)

BIN_AR = "CavityWall + InsulatedWalls_efus + FullyDblGlz_efus + AnyCooling"
CAT_AR = "C(dwtype_efus) + C(dwage_efus) + C(floor6x_efus) + C(EPceeb12e_efus)"

# ── Re-fit static references on df_ar for fair AIC comparison ────────────
print("Re-fitting static references on df_ar ...")
refs_ar = {
    "M3_ar": fit_ar("T_in ~ T_out_c + C(hour_of_day)"),
    "M5_ar": fit_ar(f"T_in ~ T_out_c + C(hour_of_day) + {BIN_AR} + {CAT_AR}"),
    "M7_ar": fit_ar(f"T_in ~ T_out_c + sin_h + cos_h + {BIN_AR} + {CAT_AR}"),
}
for name, res in refs_ar.items():
    print(f"  {name}: {'converged' if res.converged else 'NOT CONVERGED'}  AIC={res.aic:.1f}")

# ── Fit M8, M9, M10 ────────────────────────────────────────────
print("\nFitting M8, M9, M10 (dynamic LME) ...")
ar_results = {
    "M8":  fit_ar("T_in ~ T_in_lag + T_out_c + C(hour_of_day)"),
    "M9":  fit_ar(f"T_in ~ T_in_lag + T_out_c + C(hour_of_day) + {BIN_AR} + {CAT_AR}"),
    "M10": fit_ar(f"T_in ~ T_in_lag + T_out_c + sin_h + cos_h + {BIN_AR} + {CAT_AR}"),
}
for name, res in ar_results.items():
    print(f"  {name}: {'converged' if res.converged else 'NOT CONVERGED'}  AIC={res.aic:.1f}")

# ── Model comparison table ──────────────────────────────────────────────
print(f"\n{'='*92}")
print("DYNAMIC LME vs STATIC REFERENCE  (all fitted on df_ar)")
print(f"{'='*92}")
print(f"  {'Model':<8}  {'log-lik':>12}  {'AIC':>12}  {'ΔAIC vs ref':>13}  "
      f"{'φ':>8}  {'β₁_SR':>8}  {'β₁_LR':>8}")
print("  " + "-"*88)

for ref_name, dyn_name in [("M3_ar", "M8"), ("M5_ar", "M9"), ("M7_ar", "M10")]:
    ref   = refs_ar[ref_name]
    dyn   = ar_results[dyn_name]
    phi   = float(dyn.fe_params["T_in_lag"])
    b1_sr = float(dyn.fe_params["T_out_c"])
    b1_lr = b1_sr / (1 - phi)
    daic  = dyn.aic - ref.aic
    b1_ref = float(ref.fe_params["T_out_c"])
    print(f"  {ref_name:<8}  {ref.llf:>12.2f}  {ref.aic:>12.2f}  {'(reference)':>13}  "
          f"{'—':>8}  {b1_ref:>8.4f}  {'—':>8}")
    print(f"  {dyn_name:<8}  {dyn.llf:>12.2f}  {dyn.aic:>12.2f}  {daic:>+13.2f}  "
          f"{phi:>8.4f}  {b1_sr:>8.4f}  {b1_lr:>8.4f}")
    print()

# ── Random effects ───────────────────────────────────────────────────────
print(f"{'='*68}")
print("RANDOM EFFECTS — M8 / M9 / M10")
print(f"{'='*68}")
print(f"  {'Model':<6}  {'σ_u0':>8}  {'σ_u1':>8}  {'corr(u0,u1)':>12}  {'σ_ε':>8}")
print("  " + "-"*52)
for name, res in ar_results.items():
    vc   = var_components(res)
    corr = vc["sig_u01"] / (vc["sig2_u0"] * vc["sig2_u1"]) ** 0.5
    print(f"  {name:<6}  {vc['sig2_u0']**0.5:>8.4f}  {vc['sig2_u1']**0.5:>8.4f}  "
          f"{corr:>12.3f}  {vc['sig2_eps']**0.5:>8.4f}")

print("\nDone.")

### Variable coding legend

Based on EFUS 2017 interview responses (`selected_interview_responses_caseid.csv`).

#### Continuous / derived variables

| Variable | Description | Units |
|:---|:---|:---|
| `T_in` | Indoor living-room temperature | °C |
| `T_out` | Outdoor air temperature (matched by region and hour) | °C |
| `T_out_c` | $\tilde{T}_\mathrm{out} = T_\mathrm{out} - \bar{T}_\mathrm{out}$, centred on the May-August London mean (17.59 °C) | °C |
| `hour_of_day` | Hour of day $h \in \{0,1,\ldots,23\}$, where 0 = midnight | — |
| `sin_h` | $\sin(2\pi h/24)$ | — |
| `cos_h` | $\cos(2\pi h/24)$ | — |

#### Dwelling type (`dwtype_efus`)

| Code | Label |
|---:|:---|
| 1 | Detached |
| 2 | Semi-detached |
| 3 | End-terrace |
| 4 | Mid-terrace |
| 5 | Bungalow |
| 6 | Flat / maisonette |

#### Dwelling age (`dwage_efus`)

| Code | Label |
|---:|:---|
| 1 | pre-1919 |
| 2 | 1919–1944 |
| 3 | 1945–1964 |
| 4 | 1965–1974 |
| 5 | 1975–1980 |
| 6 | 1981–1990 |
| 7 | post-1990 |

#### Wall type (`WallType2x_efus`)

| Code | Label |
|---:|:---|
| 1 | Solid wall |
| 2 | Cavity wall |

#### Wall insulation (`InsulatedWalls_efus`)

| Code | Label |
|---:|:---|
| 0 | Not insulated |
| 1 | Insulated |

#### Glazing (`FullyDblGlz_efus`)

| Code | Label |
|---:|:---|
| 0 | Not fully double-glazed |
| 1 | Fully double-glazed |

#### Floor area (`floor6x_efus`)

| Code | Label |
|---:|:---|
| 1 | $< 50\,\mathrm{m}^2$ |
| 2 | $50 - 69\,\mathrm{m}^2$ |
| 3 | $70 - 89\,\mathrm{m}^2$ |
| 4 | $90 - 109\,\mathrm{m}^2$ |
| 5 | $110 - 139\,\mathrm{m}^2$ |
| 6 | $\geq 140\,\mathrm{m}^2$ |

#### EPC band (`EPceeb12e_efus`) — SAP energy efficiency rating

| Code | Band | SAP score |
|---:|:---|:---|
| 1 | C+ | $> 70$ |
| 2 | D  | $51 - 70$ |
| 3 | E  | $30 - 50$ |
| 4 | F/G | $< 30$ |

#### Cooling (`AnyCooling`)

| Code | Label |
|---:|:---|
| 0 | No mechanical cooling |
| 1 | Any mechanical cooling present |

#### Government office region (`gorEHS_efus`)

| Code | Region | Note |
|---:|:---|:---|
| 7 | London | Only region used in this notebook |
